# Notebook 03.5 — การประเมินแบบจำลองที่ดีที่สุดและการวิเคราะห์ความผิดพลาดโดยละเอียด

Notebook นี้ประเมินแบบจำลองที่ Notebook 03 ตรึงไว้ใน `configs/frozen_inference.yaml` อย่างละเอียด โดยใช้ชุด validation เป็นหลัก และไม่เปิดชุด test ใหม่ หาก Notebook 03 ประเมินชุด test ไปแล้ว (มีไฟล์ `TEST_OPENED.json`) จะนำ detection ที่บันทึกไว้มาวิเคราะห์ด้วยวิธีเดียวกันในหัวข้อสุดท้าย โดยไม่ปรับค่าใด ๆ ตามผลของชุด test

ในกรณีที่ยังไม่มีการตรึงการตั้งค่า Notebook นี้จะใช้แบบจำลองจากสูตรที่ Notebook 02 เลือก (seed 42) เป็นการชั่วคราว และกำหนด confidence threshold จากชุด validation ผลในกรณีนี้มีไว้ตรวจสอบขั้นตอนเท่านั้น และต้องรันใหม่หลังจาก Notebook 03 ตรึงการตั้งค่าแล้ว

หัวข้อของการประเมินมีดังนี้

| หัวข้อ | เนื้อหา | ชุดข้อมูลที่ใช้ | แหล่งอ้างอิงของวิธี |
|---|---|---|---|
| 1 | ข้อมูลของแบบจำลอง: สถาปัตยกรรม จำนวนพารามิเตอร์ GFLOPs สูตรการฝึก augmentation และ preprocessing | checkpoint และไฟล์การตั้งค่าของ run (ไม่ใช้ภาพ) | Mitchell et al. (2019), *FAT\** (model cards) |
| 1.1 | แผนที่ self-attention ของทุกโมดูล PSA / C3k2 (`attn=True`) ในแบบจำลอง | val 3 ภาพ (gravel, sand, mixed อย่างละ 1 ภาพ) | Ultralytics YOLO26; Wang et al. (2024), *NeurIPS* |
| 2 | พลวัตของการฝึก: loss และ metric ต่อ epoch | train (loss ของชุดฝึก) และ val (loss และ metric ระหว่างฝึก) จาก `results.csv` | — |
| 3 | metric มาตรฐานแบบ COCO พร้อมช่วงความเชื่อมั่นแบบ cluster bootstrap | val | Lin et al. (2014), *ECCV* |
| 4 | precision–recall curve, AP ตาม IoU threshold และ P/R/F1 ตาม confidence | val | Lin et al. (2014) |
| 5 | confusion matrix ระดับ instance ที่ threshold ที่ตรึงไว้ | val | — |
| 6 | การจำแนกประเภทความผิดพลาดและผลกระทบต่อ AP | val | Bolya et al. (2020), *ECCV* (TIDE) |
| 7 | คุณภาพของ mask: mask IoU และ Boundary IoU | val (เฉพาะ TP) | Cheng et al. (2021), *CVPR* |
| 8 | ผลตามขนาดของวัตถุ | val | Lin et al. (2014) |
| 9 | ความสอดคล้องของ confidence (calibration) | val | Guo et al. (2017), *ICML*; Küppers et al. (2020), *CVPR Workshops* |
| 10 | ผลรายกลุ่มของฉากและรายสถานะ | val | — |
| 11 | การจำแนกสถานะระดับภาพ (L2) | val | — |
| 12 | ภาพตัวอย่าง TP / FP / FN และภาพที่ผิดพลาดมากที่สุด | val | Hoiem et al. (2012), *ECCV* |
| 13 | ความทนทานต่อสภาพแวดล้อม (val-C) | val-C (ภาพของ val ที่ใส่การรบกวนจำลอง 3 สถานการณ์ × 5 ระดับ) | Hendrycks & Dietterich (2019), *ICLR* |
| 14 | ความเร็วในการประมวลผล | val 50 ภาพแรก | — |
| 15 | การวิเคราะห์ชุด test (เฉพาะเมื่อเปิดแล้ว) | test (detection ที่ Notebook 03 บันทึกไว้) | — |

**ข้อกำหนดในการจับคู่:** detection ถือเป็น TP เมื่อมี mask IoU กับ GT ของคลาสเดียวกันอย่างน้อย 0.5 และ GT นั้นยังไม่ถูกจับคู่ โดยจับคู่ตาม confidence จากมากไปน้อยเช่นเดียวกับ COCOeval ส่วน detection ที่ซ้อนกับ ignore region (iscrowd) เกินครึ่งหนึ่งของพื้นที่ไม่นับเป็น FP

**ชุดข้อมูลและข้อจำกัดของการตีความ:** ชุด val (131 ภาพ จาก 3 กลุ่มของฉาก) เป็นชุดเดียวกับที่ Notebook 02 และ 03 ใช้เลือก augmentation เลือกแบบจำลอง และกำหนด confidence threshold รวมถึงเกณฑ์ของสถานะ ผลในหัวข้อ 3 ถึง 14 จึงมีความลำเอียงในทางที่ดีเกินจริง (optimistic bias) และใช้เพื่อวิเคราะห์ลักษณะของความผิดพลาดเป็นหลัก ตัวเลขสำหรับรายงานประสิทธิภาพของแบบจำลองต้องมาจากชุด test (หัวข้อ 15 และ Notebook 03 หัวข้อ 8) และชุด holdout (Notebook 04) ส่วนชุด train ใช้เฉพาะในหัวข้อ 2 ผ่านค่า loss ระหว่างการฝึก

## 0. การตั้งค่า

In [ ]:
import os, sys, re, json, time, math, gzip, copy, warnings, contextlib, io, platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import yaml as _yaml
import torch
from ultralytics import YOLO
import ultralytics
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_util

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", message="A single label was found")
SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
assert (ROOT / "RESEARCH_PLAN.md").exists(), f"ROOT ไม่ถูกต้อง: {ROOT}"
CLASS_NAMES = ["gravel", "sand"]
CLASS_COLORS = {"gravel": "#0072B2", "sand": "#E69F00", "mixed": "#999999", "none": "#DDDDDD"}
KIND_COLORS = {"TP": "#0ca30c", "FP": "#d03b3b", "FN": "#fab219"}
STATES = ["none", "gravel", "sand", "mixed"]

VERSIONS = json.loads((ROOT / "manifests" / "dataset_versions.json").read_text())
DEV = VERSIONS[-1]
DEV_DIR = ROOT / "data" / f"dev_{DEV['dev_hash']}"
MANIFEST = pd.read_csv(ROOT / DEV["manifest"])
GT_VAL = DEV_DIR / "annotations" / "val_coco.json"

NOTEBOOK = "03.5_best_model_evaluation"
OUT_ROOT = ROOT / "outputs" / "03.5_best_model_evaluation"
FIG = OUT_ROOT / "figures"; TAB = OUT_ROOT / "tables"; EVAL = OUT_ROOT / "eval"
for d in (FIG, TAB, EVAL):
    d.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linewidth": 0.5})

RUN_VALC = True        # ประเมินบน val-C (ใช้ GPU ประมาณ 5–10 นาทีในครั้งแรก แล้วจึงใช้ cache)
RUN_SPEED = True       # วัดความเร็วบน GPU (ควรรันเมื่อไม่มีงานอื่นใช้ GPU)
N_BOOT = 1000

def sha256_file(p, chunk=1 << 20):
    import hashlib
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def savefig(fig, path_no_ext):
    path_no_ext = Path(path_no_ext); path_no_ext.parent.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(f"{path_no_ext}.{ext}", bbox_inches="tight")
    print(f"saved figure: {path_no_ext.relative_to(ROOT)}.png|pdf")

def save_table(df, path, index=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print(f"saved table: {path.relative_to(ROOT)} ({len(df)} rows)")

def split_images(split):
    m = MANIFEST[MANIFEST.split == split].copy()
    m["path"] = [str(DEV_DIR / split / "images" / n) for n in m.silver_name]
    return m.reset_index(drop=True)

VAL = split_images("val")
print("ROOT:", ROOT); print("val images:", len(VAL), "| groups:", VAL.group_merged.nunique())
print(VAL.state.value_counts().to_dict())

In [ ]:
# ---------- preprocessing และ attention (นิยามให้เหมือน Notebook 02/03 ทุกประการ) ----------
import torch.nn as nn
from ultralytics.nn.modules.block import C3k2

def _grayworld(img):
    f = img.astype(np.float32); m = f.reshape(-1, 3).mean(0)
    return np.clip(f * (m.mean() / np.maximum(m, 1e-6)), 0, 255).astype(np.uint8)
def _clahe(img, clip=2.0, grid=8):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB); l, a, b = cv2.split(lab)
    return cv2.cvtColor(cv2.merge([cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l), a, b]), cv2.COLOR_LAB2BGR)
def _denoise(img, h=3):
    return cv2.fastNlMeansDenoisingColored(img, None, h, h, 7, 21)
def _unsharp(img, sigma=1.0, amount=0.5):
    blur = cv2.GaussianBlur(img, (0, 0), sigma)
    return cv2.addWeighted(img, 1 + amount, blur, -amount, 0)
PREPROC_FN = {"P1_grayworld": _grayworld, "P2_clahe": _clahe, "P4_nlmeans": _denoise, "P5_unsharp": _unsharp}
def make_gamma_auto(target):
    def _gamma_auto(img):
        L = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0].mean() / 255
        g = np.log(target / 100) / np.log(max(min(L, 0.99), 0.01))
        return cv2.LUT(img, ((np.arange(256) / 255.0) ** g * 255).clip(0, 255).astype(np.uint8))
    return _gamma_auto

ATT_RECORD = {"on": False}
class ECA(nn.Module):
    def __init__(self, c, gamma=2, b=1):
        super().__init__()
        k = int(abs((math.log2(c) + b) / gamma)); k = k if k % 2 else k + 1
        self.k = k
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
        nn.init.zeros_(self.conv.weight)
        self.last_weights = None
    def forward(self, x):
        y = self.pool(x).squeeze(-1).transpose(-1, -2)
        w = 2 * torch.sigmoid(self.conv(y)).transpose(-1, -2).unsqueeze(-1)
        return x * w

class CBAMPaper(nn.Module):
    def __init__(self, c, r=16, k=7):
        super().__init__()
        h = max(c // r, 4)
        self.mlp = nn.Sequential(nn.Conv2d(c, h, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(h, c, 1, bias=False))
        nn.init.zeros_(self.mlp[2].weight)
        self.spatial = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
        nn.init.zeros_(self.spatial.weight)
        self.last_channel = None; self.last_spatial = None
    def forward(self, x):
        ca = 2 * torch.sigmoid(self.mlp(torch.amax(x, (2, 3), keepdim=True)) + self.mlp(torch.mean(x, (2, 3), keepdim=True)))
        x = x * ca
        sa = 2 * torch.sigmoid(self.spatial(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)))
        return x * sa

class C3k2Att(C3k2):
    def forward(self, x):
        return self.att(super().forward(x))

### 0.1 การเลือกแบบจำลองที่จะประเมิน

ลำดับความสำคัญคือ (1) การตั้งค่าที่ตรึงไว้ใน `configs/frozen_inference.yaml` และ (2) หากยังไม่มี ให้ใช้ run seed 42 ของสูตรที่ Notebook 02 เลือก ในกรณีที่สอง confidence threshold ของแต่ละคลาสจะกำหนดจากค่า F1 สูงสุดบนชุด validation และเกณฑ์ของสถานะจะประมาณจากชุด validation เช่นกัน

In [ ]:
FROZEN_PATH = ROOT / "configs" / "frozen_inference.yaml"
SEL_PATH = ROOT / "configs" / "aug_recipe_selected.json"
SEL = json.loads(SEL_PATH.read_text()) if SEL_PATH.exists() else None
if FROZEN_PATH.exists():
    FROZEN = _yaml.safe_load(FROZEN_PATH.read_text())
    WEIGHTS = ROOT / FROZEN["weights"]
    assert sha256_file(WEIGHTS) == FROZEN["weights_sha256"], "checkpoint does not match frozen config"
    PROVISIONAL = False
    MODEL_NAME = FROZEN["selected"]
    PP_NAME = FROZEN.get("preprocessing")
else:
    assert SEL is not None, "ต้องรัน Notebook 02 ให้ครบก่อน"
    FROZEN = None
    WEIGHTS = ROOT / SEL["run_dirs"]["42"] / "weights" / "best.pt"
    PROVISIONAL = True
    MODEL_NAME = SEL["arm"] + " (provisional, seed 42)"
    PP_NAME = SEL.get("preprocessing")
RUN_DIR = WEIGHTS.parent.parent
_pp_par = ((FROZEN or {}).get("preprocessing_params") or (SEL or {}).get("preprocessing_params") or {})
if _pp_par.get("TARGET_L") is not None:
    PREPROC_FN["P3_gamma_auto"] = make_gamma_auto(_pp_par["TARGET_L"])
TF = PREPROC_FN.get(PP_NAME) if PP_NAME else None
IMGSZ = int(FROZEN.get("imgsz", 640)) if FROZEN else 640
assert PP_NAME is None or TF is not None, f"preprocessing {PP_NAME} must be re-implemented here identically to Notebook 02"
STATE_MIN_CONF = FROZEN.get("state_min_conf", 0.05) if FROZEN else 0.05
EVAL_TAG = re.sub(r"[^A-Za-z0-9]+", "_", MODEL_NAME).strip("_") + "_" + sha256_file(WEIGHTS)[:8]
print(json.dumps(dict(model=MODEL_NAME, provisional=PROVISIONAL, imgsz=IMGSZ, model_size=(FROZEN or {}).get("model_size", "n"), weights=str(WEIGHTS.relative_to(ROOT)),
                      run_dir=str(RUN_DIR.relative_to(ROOT)), preprocessing=PP_NAME, eval_tag=EVAL_TAG), indent=2))
if PROVISIONAL:
    print("\nหมายเหตุ: ยังไม่มีการตรึงการตั้งค่า ผลใน Notebook นี้เป็นผลชั่วคราว")

## 1. ข้อมูลของแบบจำลอง

**ชุดข้อมูล:** ไม่ใช้ภาพ อ่านจาก checkpoint, `args.yaml`, `DONE.json` และ `configs/aug_recipe_selected.json` จำนวนภาพของชุดฝึกนับจากชุดข้อมูลที่ใช้ฝึกจริงหลัง offline augmentation


ส่วนนี้สรุปรายละเอียดที่จำเป็นต่อการทำซ้ำตามแนวคิดของ model card ได้แก่ สถาปัตยกรรมรายชั้น จำนวนพารามิเตอร์ ปริมาณการคำนวณ ข้อมูลที่ใช้ฝึก สูตรการฝึก augmentation preprocessing และค่าที่ตรึงไว้สำหรับการใช้งาน

In [ ]:
model = YOLO(str(WEIGHTS))
net = model.model
n_par = sum(p.numel() for p in net.parameters())
try:
    from ultralytics.utils.torch_utils import get_flops
    gflops = float(get_flops(net, IMGSZ))
except Exception as e:
    gflops = float("nan"); print("GFLOPs not available:", e)
rows = []
for m in net.model:
    att = getattr(m, "att", None)
    rows.append(dict(idx=m.i, from_=str(m.f), module=type(m).__name__ + (f"+{type(att).__name__}" if att is not None else ""),
                     params=sum(p.numel() for p in m.parameters()), out_channels=next((mm.out_channels for mm in reversed(list(m.modules()))
                                                                                         if isinstance(mm, nn.Conv2d)), None)))
LAYERS = pd.DataFrame(rows)
save_table(LAYERS, TAB / "model_layers.csv")

args = _yaml.safe_load((RUN_DIR / "args.yaml").read_text()) if (RUN_DIR / "args.yaml").exists() else {}
done = json.loads((RUN_DIR / "DONE.json").read_text()) if (RUN_DIR / "DONE.json").exists() else {}
data_yaml = Path(args.get("data", done.get("data", "")))
n_train = len(list((data_yaml.parent / "train" / "images").glob("*"))) if data_yaml.exists() else None
wall = (RUN_DIR / "wall_time_s.txt").read_text().strip() if (RUN_DIR / "wall_time_s.txt").exists() else None
CARD = {
    "model": MODEL_NAME, "provisional": PROVISIONAL, "framework": f"ultralytics {ultralytics.__version__}",
    "architecture": type(net).__name__, "base_weights": args.get("model"), "task": args.get("task"),
    "parameters": int(n_par), "GFLOPs@imgsz": round(gflops, 2), "imgsz_eval": IMGSZ, "model_size": (FROZEN or {}).get("model_size", "n"), "round": (FROZEN or {}).get("round", 1), "layers": len(LAYERS),
    "attention": (FROZEN or {}).get("attention"), "imgsz": args.get("imgsz"),
    "train_images (after offline aug)": n_train, "val_images": len(VAL), "data": str(data_yaml),
    "epochs": args.get("epochs"), "batch": args.get("batch"), "optimizer": args.get("optimizer"), "lr0": args.get("lr0"),
    "lrf": args.get("lrf"), "weight_decay": args.get("weight_decay"), "warmup_epochs": args.get("warmup_epochs"),
    "close_mosaic": args.get("close_mosaic"), "amp": args.get("amp"), "seed": args.get("seed"), "train_wall_time_s": wall,
    "online_augmentation": {k: args.get(k) for k in ("hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "shear",
                                                      "perspective", "flipud", "fliplr", "mosaic", "mixup", "copy_paste", "copy_paste_mode")},
    "offline_augmentation_components": (SEL or {}).get("passed_components"),
    "preprocessing": PP_NAME,
    "conf_thresholds": (FROZEN or {}).get("conf_thresholds"), "state_thresholds": (FROZEN or {}).get("state_thresholds"),
    "weights_sha256": sha256_file(WEIGHTS), "dev_hash": DEV["dev_hash"],
}
(OUT_ROOT / "model_card.json").write_text(json.dumps(CARD, indent=2, default=str))
print(json.dumps(CARD, indent=2, default=str))

In [ ]:
# พารามิเตอร์รายชั้น (แยก backbone / neck / head ตามตำแหน่งชั้นใน yaml ของ YOLO26-seg)
head_idx = LAYERS.idx.max()
LAYERS["part"] = np.where(LAYERS.idx <= 10, "backbone", np.where(LAYERS.idx < head_idx, "neck", "head"))
part_col = {"backbone": "#2a78d6", "neck": "#1baf7a", "head": "#eb6834"}
fig, ax = plt.subplots(figsize=(8.5, 3.0))
ax.bar(LAYERS.idx, LAYERS.params / 1e3, color=[part_col[p] for p in LAYERS.part], width=0.8)
for i, r in LAYERS.iterrows():
    ax.text(r.idx, r.params / 1e3, r.module, rotation=90, fontsize=5.5, ha="center", va="bottom", color="#52514e")
ax.set_xlabel("layer index"); ax.set_ylabel("parameters (thousand)"); ax.set_xticks(LAYERS.idx)
ax.tick_params(axis="x", labelsize=6)
ax.legend(handles=[Rectangle((0, 0), 1, 1, color=c, label=k) for k, c in part_col.items()], frameon=False, fontsize=7)
ax.set_title(f"Parameters per layer — total {n_par/1e6:.2f} M, {gflops:.1f} GFLOPs @{IMGSZ}")
ax.set_ylim(0, LAYERS.params.max() / 1e3 * 1.45)
fig.tight_layout(); savefig(fig, FIG / "fig01_parameters_per_layer"); plt.show()
display(LAYERS.groupby("part").params.sum().rename("params").to_frame().assign(share=lambda d: d.params / d.params.sum()).round(3))

### 1.1 แผนที่ self-attention ของแบบจำลอง

**ชุดข้อมูล:** val 3 ภาพ ได้แก่ภาพแรกของสถานะ gravel, sand และ mixed ตามลำดับ image_id ใช้ preprocessing ชุดเดียวกับตอนฝึก

แบบจำลอง YOLO26n-seg มี self-attention ในโมดูล `PSABlock` ซึ่งอยู่ใน `C2PSA` (ชั้น 10) และใน `C3k2` ที่ตั้ง `attn=True` (ชั้น 22 ในสถาปัตยกรรมต้นฉบับ และชั้นอื่นตามการตั้งค่าที่ตรึงไว้จาก Notebook 03) ส่วนนี้คำนวณ $\mathrm{softmax}(QK^\top/\sqrt{d_k})$ ซ้ำจาก input จริงของทุกโมดูล `Attention` แล้วแสดง attention ที่แต่ละตำแหน่งได้รับโดยเฉลี่ย แผนที่ attention ของ query ที่อยู่ลึกที่สุดในวัสดุแต่ละคลาสและในพื้นหลัง และขนาดของ output ของ attention เทียบกับ input ซึ่งบอกว่า attention มีส่วนต่อ feature มากเพียงใด ค่า entropy ที่ normalize ด้วย log N ใกล้ 1 หมายถึงการกระจาย attention ทั่วภาพ และค่าต่ำหมายถึงการเพ่งเฉพาะบริเวณ

In [ ]:
# ---------- การดึงค่า attention ของ PSABlock (C3k2 attn=True และ C2PSA) เพื่อแสดงผล ----------
from ultralytics.nn.modules.block import Attention as UAttention
from ultralytics.data.augment import LetterBox as _LetterBox

def capture_psa(seg_model, x):
    # คำนวณ softmax(QK^T) ซ้ำจาก input จริงของโมดูล Attention ด้วยสูตรเดียวกับ ultralytics (ไม่แก้ forward ของแบบจำลอง)
    rec, hooks = {}, []
    for m in seg_model.model:
        for name, sub in m.named_modules():
            if isinstance(sub, UAttention):
                def hk(mod, inp, out, key=f"{m.i:02d}_{type(m).__name__}.{name}", li=m.i):
                    xx = inp[0]; B, C, H, W = xx.shape; N = H * W
                    q, k, v = mod.qkv(xx).view(B, mod.num_heads, mod.key_dim * 2 + mod.head_dim, N).split(
                        [mod.key_dim, mod.key_dim, mod.head_dim], dim=2)
                    a = ((q * mod.scale).transpose(-2, -1) @ k).softmax(dim=-1)[0].float().cpu()
                    rec[key] = dict(layer=li, attn=a, H=H, W=W, x_in=xx[0].float().cpu(), branch=out[0].float().cpu())
                hooks.append(sub.register_forward_hook(hk))
    seg_model.eval()
    with torch.no_grad():
        seg_model(x)
    for h in hooks:
        h.remove()
    return rec

def letterbox_masks(class_masks, valid_shape, size=640):
    # แปลง mask ของแต่ละคลาสด้วย LetterBox เดียวกับภาพ; ค่า padding (114) ต่ำกว่า 127 จึงไม่ถูกนับเป็นวัสดุ
    lb = lambda m: _LetterBox(new_shape=(size, size), auto=False)(image=np.repeat((m * 255).astype(np.uint8)[..., None], 3, 2))[..., 0] > 127
    out = {c: lb(m) for c, m in class_masks.items() if m.sum() > 0}
    valid = lb(np.ones(valid_shape, np.uint8))
    return out, valid

def pick_queries(lbm, valid):
    # จุด query: ตำแหน่งที่อยู่ลึกที่สุดในวัสดุแต่ละคลาส และตำแหน่งพื้นหลังที่ห่างจากวัสดุมากที่สุด (ไม่รวม padding)
    pts = {}
    for c, m in lbm.items():
        d = cv2.distanceTransform(m.astype(np.uint8), cv2.DIST_L2, 5); y, x_ = np.unravel_index(d.argmax(), d.shape); pts[c] = (int(x_), int(y))
    union = np.zeros(valid.shape, bool)
    for m in lbm.values():
        union |= m
    d = cv2.distanceTransform((~union & valid).astype(np.uint8), cv2.DIST_L2, 5); y, x_ = np.unravel_index(d.argmax(), d.shape)
    pts["background"] = (int(x_), int(y))
    return pts

QCOL = {"gravel": "#0072B2", "sand": "#E69F00", "background": "#ffffff"}

def plot_psa(rec, lb_rgb, pts, title, path):
    keys = sorted(rec); size = lb_rgb.shape[0]
    ncol = 2 + len(pts) + 1
    fig, axes = plt.subplots(len(keys), ncol, figsize=(2.05 * ncol, 2.25 * len(keys)), squeeze=False)
    rows = []
    for r, key in enumerate(keys):
        d = rec[key]; A = d["attn"].mean(0); H, W = d["H"], d["W"]; s = size / H
        recv = A.mean(0).reshape(H, W).numpy()
        ent = float((-(d["attn"] * (d["attn"] + 1e-12).log()).sum(-1)).mean() / math.log(H * W))
        ratio = float(d["branch"].norm() / max(d["x_in"].norm(), 1e-12))
        up = lambda z: cv2.resize(z, (size, size), interpolation=cv2.INTER_NEAREST)
        ax = axes[r, 0]; ax.imshow(lb_rgb)
        for nm, (px, py) in pts.items():
            ax.plot(px, py, "o", ms=6, mfc=QCOL[nm], mec="#0b0b0b", mew=1)
        ax.set_ylabel(f"layer {d['layer']}\n{key.split('.', 1)[1]}", fontsize=6)
        ax.set_title("image and query points", fontsize=6.5)
        ax = axes[r, 1]; ax.imshow(lb_rgb); ax.imshow(up(recv), cmap="magma", alpha=0.6)
        ax.set_title(f"mean attention received\n{H}x{W} tokens, {d['attn'].shape[0]} head(s)", fontsize=6.5)
        for j, (nm, (px, py)) in enumerate(pts.items()):
            qi = min(int(py // s), H - 1) * W + min(int(px // s), W - 1)
            ax = axes[r, 2 + j]; ax.imshow(lb_rgb); ax.imshow(up(A[qi].reshape(H, W).numpy()), cmap="magma", alpha=0.6)
            ax.plot(px, py, "o", ms=5, mfc=QCOL[nm], mec="#0b0b0b", mew=1); ax.set_title(f"where a {nm} token looks", fontsize=6.5)
        ax = axes[r, -1]; ax.imshow(d["branch"].abs().mean(0).numpy(), cmap="viridis")
        ax.set_title(f"|attention output| (mean over ch)\n‖branch‖/‖input‖ {ratio:.2f}, entropy {ent:.2f}", fontsize=6.5)
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([]); a.grid(False)
        rows.append(dict(module=key, layer=d["layer"], tokens=f"{H}x{W}", heads=int(d["attn"].shape[0]), branch_to_input_norm=ratio, norm_entropy=ent))
    fig.suptitle(title, fontsize=8); fig.tight_layout(); savefig(fig, path); plt.show()
    return pd.DataFrame(rows)

# section_1_1_attention
dev_ = "cuda" if torch.cuda.is_available() else "cpu"
att_model = YOLO(str(WEIGHTS)).model.float().to(dev_).eval()
with contextlib.redirect_stdout(io.StringIO()):
    gt_v = COCO(str(GT_VAL))
ATT_STATS = []
for st in ("gravel", "sand", "mixed"):
    sub_ = VAL[VAL.state == st]
    if sub_.empty:
        continue
    r = sub_.iloc[0]; im0 = cv2.imread(r.path); im0 = TF(im0) if TF else im0; h0, w0 = im0.shape[:2]
    lb = _LetterBox(new_shape=(IMGSZ, IMGSZ), auto=False)(image=im0)
    xt = torch.from_numpy(lb[..., ::-1].copy()).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev_)
    cm = {c: np.zeros((h0, w0), np.uint8) for c in CLASS_NAMES}
    for a in gt_v.imgToAnns[int(r.image_id)]:
        if not a.get("iscrowd", 0):
            cm[CLASS_NAMES[a["category_id"] - 1]] |= gt_v.annToMask(a)
    lbm, valid = letterbox_masks(cm, (h0, w0), size=IMGSZ); pts = pick_queries(lbm, valid)
    rec = capture_psa(att_model, xt)
    t = plot_psa(rec, cv2.cvtColor(lb, cv2.COLOR_BGR2RGB), pts, f"{MODEL_NAME}: self-attention on a {st} image ({r.group_merged[:8]}, frame {r.frame})",
                 FIG / f"fig01b_attention_{st}")
    ATT_STATS.append(t.assign(image_state=st, image_id=int(r.image_id))); del rec
ATT_STATS = pd.concat(ATT_STATS); save_table(ATT_STATS, TAB / "attention_stats.csv")
display(ATT_STATS.round(3))
del att_model; torch.cuda.empty_cache()

## 2. พลวัตของการฝึก

**ชุดข้อมูล:** loss ของชุดฝึก (train หลัง offline augmentation และ preprocessing) และ loss กับ metric ของชุด val ที่ Ultralytics คำนวณทุก epoch จากไฟล์ `results.csv` ของ run ที่ตรึงไว้


กราฟแสดง loss ของชุดฝึกและชุด validation แยกตามองค์ประกอบ (box, segmentation, classification) และ metric ที่ Ultralytics คำนวณระหว่างการฝึก ค่า metric ในส่วนนี้ใช้ตัวประเมินของ Ultralytics จึงอาจต่างจากค่าที่คำนวณด้วย pycocotools ในหัวข้อ 3 เล็กน้อย

In [ ]:
RES = pd.read_csv(RUN_DIR / "results.csv"); RES.columns = [c.strip() for c in RES.columns]
save_table(RES, TAB / "training_results.csv")
loss_keys = [k for k in ("box_loss", "seg_loss", "cls_loss", "sem_loss") if f"train/{k}" in RES]
fig, axes = plt.subplots(1, len(loss_keys) + 2, figsize=(2.6 * (len(loss_keys) + 2), 2.6))
for ax, k in zip(axes, loss_keys):
    ax.plot(RES.epoch, RES[f"train/{k}"], color="#2a78d6", lw=2, label="train")
    if f"val/{k}" in RES and RES[f"val/{k}"].abs().sum() > 0:
        ax.plot(RES.epoch, RES[f"val/{k}"], color="#eb6834", lw=2, label="val")
    ax.set_title(k); ax.set_xlabel("epoch")
axes[0].legend(frameon=False, fontsize=7)
ax = axes[len(loss_keys)]
for col, lab, c in (("metrics/mAP50-95(M)", "mask mAP50-95", "#2a78d6"), ("metrics/mAP50(M)", "mask mAP50", "#1baf7a"),
                    ("metrics/mAP50-95(B)", "box mAP50-95", "#eb6834")):
    if col in RES:
        ax.plot(RES.epoch, RES[col], lw=2, color=c, label=lab)
best_ep = int(RES.loc[RES["metrics/mAP50-95(M)"].idxmax(), "epoch"]) if "metrics/mAP50-95(M)" in RES else None
if best_ep:
    ax.axvline(best_ep, color="#52514e", ls=":", lw=1); ax.text(best_ep, ax.get_ylim()[1], f" best ep {best_ep}", fontsize=6.5, va="top")
ax.set_title("val metrics (Ultralytics)"); ax.set_xlabel("epoch"); ax.legend(frameon=False, fontsize=6.5, loc="lower left")
ax = axes[-1]
for col, c in (("lr/pg0", "#2a78d6"),):
    if col in RES:
        ax.plot(RES.epoch, RES[col], lw=2, color=c)
ax.set_title("learning rate"); ax.set_xlabel("epoch")
fig.tight_layout(); savefig(fig, FIG / "fig02_training_curves"); plt.show()

## 3. Metric มาตรฐานแบบ COCO

**ชุดข้อมูล:** val ทั้ง 131 ภาพ ใช้ preprocessing ชุดเดียวกับตอนฝึก


ทำนายด้วย confidence ต่ำสุด 0.001 และ `max_det=100` ตามแบบแผนของ COCO แล้วคำนวณ mask AP และ box AP ด้วย pycocotools ทั้งแบบที่ไม่นับ ignore region (protocol หลัก) และแบบ strict ที่นับ ignore region เป็น GT ปกติ ช่วงความเชื่อมั่น 95% คำนวณด้วย cluster bootstrap โดยสุ่มกลุ่มของฉาก (`group_merged`) แทนการสุ่มภาพ เพื่อไม่ให้ภาพจากวิดีโอเดียวกันทำให้ช่วงแคบเกินจริง

In [ ]:
def load_gt(json_path, strict=False):
    d = json.loads(Path(json_path).read_text())
    if strict:
        for a in d["annotations"]:
            a["iscrowd"] = 0
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(); gt.dataset = d; gt.createIndex()
    return gt

def predict_dets(weights, records, imgsz=640, conf=0.001, max_det=100, batch=8, transform=None, desc=""):
    m = YOLO(str(weights)); dets, t0 = [], time.time()
    recs = records.to_dict("records")
    for k in range(0, len(recs), batch):
        chunk = recs[k:k + batch]
        imgs = [cv2.imread(r["path"]) for r in chunk]
        if transform is not None:
            imgs = [transform(im) for im in imgs]
        res = m.predict(imgs, imgsz=imgsz, conf=conf, max_det=max_det, retina_masks=True, verbose=False)
        for r, out in zip(chunk, res):
            if out.masks is None or len(out.boxes) == 0:
                continue
            ms = out.masks.data.cpu().numpy().astype(np.uint8)
            xyxy = out.boxes.xyxy.cpu().numpy(); cls = out.boxes.cls.cpu().numpy().astype(int); sc = out.boxes.conf.cpu().numpy()
            for mm, b, c, s in zip(ms, xyxy, cls, sc):
                if mm.shape != (r["height"], r["width"]):
                    mm = cv2.resize(mm, (r["width"], r["height"]), interpolation=cv2.INTER_NEAREST)
                rle = mask_util.encode(np.asfortranarray(mm)); rle["counts"] = rle["counts"].decode()
                dets.append(dict(image_id=int(r["image_id"]), category_id=int(c) + 1, score=float(s), segmentation=rle,
                                 bbox=[float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])], area=float(mm.sum())))
    print(f"predicted {len(recs)} images -> {len(dets)} detections in {time.time()-t0:.1f}s {desc}")
    return dets

def cached_dets(cache_file, fn):
    cache_file = Path(cache_file)
    if cache_file.exists():
        with gzip.open(cache_file, "rt") as f:
            return json.load(f)
    dets = fn(); cache_file.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(cache_file, "wt") as f:
        json.dump(dets, f)
    return dets

def run_cocoeval(gt, dets, iou_type="segm", img_ids=None):
    dts = [dict(d) for d in dets if img_ids is None or d["image_id"] in set(img_ids)]
    if iou_type == "bbox":
        for d in dts:
            d.pop("segmentation", None)
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(dts) if dts else COCO()
        E = COCOeval(gt, dt, iou_type)
        if img_ids is not None:
            E.params.imgIds = sorted(img_ids)
        E.evaluate(); E.accumulate(); E.summarize()
    return E

STAT_NAMES = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]

def image_records(E):
    recs = {}
    a_all, md_ = E.params.areaRng[0], E.params.maxDets[-1]
    for e in E.evalImgs:
        if e is None or e["aRng"] != a_all or e["maxDet"] != md_:
            continue
        recs[(e["image_id"], e["category_id"])] = dict(scores=np.asarray(e["dtScores"], float), matched=np.asarray(e["dtMatches"]) > 0,
                                                       dt_ig=np.asarray(e["dtIgnore"]).astype(bool), npig=int((np.asarray(e["gtIgnore"]) == 0).sum()))
    return recs

def ap_from_records(recs, img_ids, cat_id, n_iou=10):
    rs = [recs[(i, cat_id)] for i in img_ids if (i, cat_id) in recs]
    if not rs:
        return float("nan")
    npig = sum(r["npig"] for r in rs)
    if npig == 0:
        return float("nan")
    scores = np.concatenate([r["scores"] for r in rs])
    if len(scores) == 0:
        return 0.0
    order = np.argsort(-scores, kind="mergesort")
    M = np.concatenate([r["matched"] for r in rs], axis=1)[:, order]
    IG = np.concatenate([r["dt_ig"] for r in rs], axis=1)[:, order]
    rec_thr = np.linspace(0, 1, 101); aps = []
    for t in range(n_iou):
        tp = np.cumsum(M[t] & ~IG[t]); fp = np.cumsum(~M[t] & ~IG[t])
        rc = tp / npig; pr = tp / np.maximum(tp + fp, np.spacing(1))
        pr = np.maximum.accumulate(pr[::-1])[::-1]
        idx = np.searchsorted(rc, rec_thr, side="left")
        aps.append(np.where(idx < len(pr), pr[np.minimum(idx, len(pr) - 1)], 0.0).mean())
    return float(np.mean(aps))

def operating_point(recs, img_ids, cat_id, iou_index=0, thresholds=None):
    rs = [recs[(i, cat_id)] for i in img_ids if (i, cat_id) in recs]
    npig = sum(r["npig"] for r in rs)
    s = np.concatenate([r["scores"] for r in rs]) if rs else np.zeros(0)
    m = np.concatenate([r["matched"][iou_index] for r in rs]) if rs else np.zeros(0, bool)
    ig = np.concatenate([r["dt_ig"][iou_index] for r in rs]) if rs else np.zeros(0, bool)
    thresholds = np.round(np.arange(0.01, 0.96, 0.01), 2) if thresholds is None else np.atleast_1d(thresholds)
    rows = []
    for th in thresholds:
        k = (s >= th) & ~ig
        tp = int((m & k).sum()); fp = int((~m & k).sum()); fn = npig - tp
        p = tp / max(tp + fp, 1); r_ = tp / max(npig, 1)
        rows.append(dict(conf=float(th), TP=tp, FP=fp, FN=fn, precision=p, recall=r_, F1=2 * p * r_ / max(p + r_, 1e-9)))
    return pd.DataFrame(rows)

def cluster_bootstrap(records, stat_fn, n_boot=N_BOOT, seed=SEED, group_col="group_merged"):
    rng = np.random.default_rng(seed)
    groups = records[group_col].unique()
    by_g = {g: records.image_id[records[group_col] == g].astype(int).tolist() for g in groups}
    vals = np.asarray([stat_fn([i for g in rng.choice(groups, size=len(groups), replace=True) for i in by_g[g]]) for _ in range(n_boot)], float)
    return dict(ci_low=float(np.nanpercentile(vals, 2.5)), ci_high=float(np.nanpercentile(vals, 97.5)), n_clusters=len(groups))

In [ ]:
def coco_block(gt_json, dets, records, tag):
    gt = load_gt(gt_json); gt_s = load_gt(gt_json, strict=True)
    ids = records.image_id.astype(int).tolist()
    out, rows = {}, []
    for it, lab in (("segm", "mask"), ("bbox", "box")):
        E = run_cocoeval(gt, dets, it, ids); Es = run_cocoeval(gt_s, dets, it, ids)
        out[it] = dict(E=E, recs=image_records(E))
        for k, v in zip(STAT_NAMES, E.stats):
            rows.append(dict(type=lab, protocol="ignore", metric=k, value=float(v)))
        for k, v in zip(STAT_NAMES, Es.stats):
            rows.append(dict(type=lab, protocol="strict", metric=k, value=float(v)))
        p = E.eval["precision"]
        for kk, cid in enumerate(E.params.catIds):
            for t_name, t_sl in (("AP", slice(None)), ("AP50", 0), ("AP75", 5)):
                pr = p[t_sl, :, kk, 0, -1]
                rows.append(dict(type=lab, protocol="ignore", metric=f"{t_name}_{CLASS_NAMES[cid-1]}",
                                 value=float(np.mean(pr[pr > -1])) if (pr > -1).any() else float("nan")))
    T = pd.DataFrame(rows); T["ci_low"] = np.nan; T["ci_high"] = np.nan
    recs = out["segm"]["recs"]
    ci = cluster_bootstrap(records, lambda ii: np.nanmean([ap_from_records(recs, ii, c) for c in (1, 2)]))
    ci_c = {c: cluster_bootstrap(records, lambda ii, c=c: ap_from_records(recs, ii, c)) for c in (1, 2)}
    T.loc[(T.type == "mask") & (T.protocol == "ignore") & (T.metric == "AP"), ["ci_low", "ci_high"]] = [ci["ci_low"], ci["ci_high"]]
    for c in (1, 2):
        T.loc[(T.type == "mask") & (T.protocol == "ignore") & (T.metric == f"AP_{CLASS_NAMES[c-1]}"), ["ci_low", "ci_high"]] = [ci_c[c]["ci_low"], ci_c[c]["ci_high"]]
    save_table(T, TAB / f"{tag}_coco_metrics.csv")
    return out, T

DETS_VAL = cached_dets(EVAL / f"dets_val_{EVAL_TAG}.json.gz", lambda: predict_dets(WEIGHTS, VAL, imgsz=IMGSZ, transform=TF, desc="val"))
COCO_VAL, T_VAL = coco_block(GT_VAL, DETS_VAL, VAL, "val")
piv = T_VAL[T_VAL.protocol == "ignore"].pivot_table(index="metric", columns="type", values="value").reindex(
    STAT_NAMES + [f"{a}_{c}" for c in CLASS_NAMES for a in ("AP", "AP50", "AP75")])
display(piv.round(4))
display(T_VAL.dropna(subset=["ci_low"])[["metric", "value", "ci_low", "ci_high"]].round(4))
display(T_VAL[(T_VAL.metric.isin(["AP", "AP50", "AP75"]))].pivot_table(index=["type", "metric"], columns="protocol", values="value").round(4))

### 3.1 Confidence threshold ที่ใช้ในการวิเคราะห์ระดับ instance

**ชุดข้อมูล:** val (threshold ถูกกำหนดจาก val ใน Notebook 03)


หัวข้อ 5 ถึง 12 วิเคราะห์ที่ threshold ของการใช้งานจริง ถ้ามีการตรึงการตั้งค่าแล้วจะใช้ค่าจาก `frozen_inference.yaml` มิฉะนั้นจะเลือก threshold ที่ให้ F1 สูงสุดที่ IoU 0.5 บนชุด validation

In [ ]:
ids_val = VAL.image_id.astype(int).tolist()
OPS = {c: operating_point(COCO_VAL["segm"]["recs"], ids_val, i + 1) for i, c in enumerate(CLASS_NAMES)}
if FROZEN is not None:
    CONF_TH = {c: float(FROZEN["conf_thresholds"][c]) for c in CLASS_NAMES}
else:
    CONF_TH = {c: float(OPS[c].sort_values("F1").iloc[-1].conf) for c in CLASS_NAMES}
print("confidence thresholds:", CONF_TH, "(frozen)" if FROZEN is not None else "(F1-optimal on val, provisional)")
display(pd.DataFrame([dict(cls=c, **operating_point(COCO_VAL["segm"]["recs"], ids_val, i + 1, thresholds=[CONF_TH[c]]).iloc[0].to_dict())
                      for i, c in enumerate(CLASS_NAMES)]).round(4))

## 4. Precision–recall curve และ metric ตาม threshold

**ชุดข้อมูล:** val


กราฟซ้ายแสดง precision–recall curve ของแต่ละคลาสที่ IoU 0.5 และ 0.75 จากตัวแปร `precision` ของ COCOeval (101 จุดของ recall) จุดสีดำคือ operating point ที่ใช้งาน กราฟกลางแสดง AP เป็นฟังก์ชันของ IoU threshold ตั้งแต่ 0.50 ถึง 0.95 ซึ่งบอกว่า AP ลดลงเร็วเพียงใดเมื่อต้องการขอบเขตของ mask ที่แม่นยำขึ้น และกราฟขวาแสดง precision, recall และ F1 ตาม confidence threshold

In [ ]:
def pr_figure(coco, recs, ids, conf_th, tag):
    E = coco["segm"]["E"]; p = E.eval["precision"]; rthr = E.params.recThrs; iouT = E.params.iouThrs
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.0))
    ax = axes[0]
    for kk, cid in enumerate(E.params.catIds):
        c = CLASS_NAMES[cid - 1]
        for ti, ls in ((0, "-"), (5, "--")):
            pr = p[ti, :, kk, 0, -1]; pr = np.where(pr < 0, np.nan, pr)
            ax.plot(rthr, pr, ls=ls, lw=2, color=CLASS_COLORS[c], label=f"{c} IoU{iouT[ti]:.2f} (AP {np.nanmean(pr):.3f})")
        o = operating_point(recs, ids, cid, thresholds=[conf_th[c]]).iloc[0]
        ax.plot(o.recall, o.precision, "o", ms=8, mfc=CLASS_COLORS[c], mec="#0b0b0b", mew=1.2)
        ax.annotate(f"conf {conf_th[c]:.2f}", (o.recall, o.precision), textcoords="offset points", xytext=(4, -10), fontsize=6.5)
    ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_title("mask PR curve"); ax.legend(frameon=False, fontsize=6)
    ax = axes[1]
    for kk, cid in enumerate(E.params.catIds):
        c = CLASS_NAMES[cid - 1]
        ap_t = [np.nanmean(np.where(p[t, :, kk, 0, -1] < 0, np.nan, p[t, :, kk, 0, -1])) for t in range(len(iouT))]
        ax.plot(iouT, ap_t, "-o", ms=4, lw=2, color=CLASS_COLORS[c], label=c)
    for itype, ls in (("bbox", ":"),):
        Eb = coco[itype]["E"]; pb = Eb.eval["precision"]
        ap_b = [np.nanmean(np.where(pb[t, :, :, 0, -1] < 0, np.nan, pb[t, :, :, 0, -1])) for t in range(len(iouT))]
        ax.plot(iouT, ap_b, ls=ls, lw=1.5, color="#52514e", label="box (mean)")
    ax.set_xlabel("IoU threshold"); ax.set_ylabel("AP"); ax.set_ylim(0, 1); ax.set_title("AP vs IoU threshold"); ax.legend(frameon=False, fontsize=6.5)
    for ax, c in zip(axes[2:], CLASS_NAMES):
        o = operating_point(recs, ids, CLASS_NAMES.index(c) + 1)
        for col, colr in (("precision", "#2a78d6"), ("recall", "#eb6834"), ("F1", "#1baf7a")):
            ax.plot(o.conf, o[col], lw=2, color=colr, label=col)
        ax.axvline(conf_th[c], color="#0b0b0b", ls=":", lw=1)
        ax.set_xlabel("confidence threshold"); ax.set_ylim(0, 1); ax.set_title(f"{c}: P / R / F1 vs confidence")
        ax.legend(frameon=False, fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / f"fig03_{tag}_pr_curves"); plt.show()
    pd.concat([operating_point(recs, ids, i + 1).assign(cls=c) for i, c in enumerate(CLASS_NAMES)]).pipe(save_table, TAB / f"{tag}_prf_vs_conf.csv")

pr_figure(COCO_VAL, COCO_VAL["segm"]["recs"], ids_val, CONF_TH, "val")

## 5. การจับคู่ระดับ instance และ confusion matrix

**ชุดข้อมูล:** val ที่ threshold ของหัวข้อ 3.1


ทุก detection ได้รับการจับคู่กับ GT ของภาพเดียวกัน ผลลัพธ์ของขั้นนี้ใช้ในหัวข้อ 5 ถึง 12 ทั้งหมด confusion matrix แบบนี้จับคู่โดยไม่คำนึงถึงคลาสก่อน (IoU ≥ 0.5 ตาม confidence จากมากไปน้อย) แล้วนับคู่ (คลาสจริง, คลาสที่ทำนาย) ส่วน detection ที่ไม่ได้จับคู่นับในแถว background (FP) และ GT ที่ไม่ได้จับคู่นับในคอลัมน์ background (FN) ซึ่งเป็นรูปแบบเดียวกับที่ Ultralytics ใช้ แต่ใช้ threshold ที่ตรึงไว้และ mask IoU

In [ ]:
def build_matches(gt_json, dets, records, min_score=0.001):
    # คืนตารางราย detection และราย GT พร้อม IoU ที่จำเป็นต่อการจำแนกความผิดพลาด
    gt = load_gt(gt_json)
    D_rows, G_rows, cache = [], [], {}
    by_img = defaultdict(list)
    for k, d in enumerate(dets):
        if d["score"] >= min_score:
            by_img[d["image_id"]].append((k, d))
    for r in records.itertuples():
        iid = int(r.image_id)
        anns = gt.imgToAnns[iid]
        G = [a for a in anns if not a.get("iscrowd", 0)]; IG = [a for a in anns if a.get("iscrowd", 0)]
        g_rle = [gt.annToRLE(a) for a in G]; ig_rle = [gt.annToRLE(a) for a in IG]
        ds = sorted(by_img[iid], key=lambda x: -x[1]["score"])
        d_rle = [d["segmentation"] for _, d in ds]
        iou = mask_util.iou(d_rle, g_rle, [0] * len(g_rle)) if d_rle and g_rle else np.zeros((len(d_rle), len(g_rle)))
        ioa = mask_util.iou(d_rle, ig_rle, [1] * len(ig_rle)) if d_rle and ig_rle else np.zeros((len(d_rle), 0))
        iou = np.asarray(iou).reshape(len(d_rle), len(g_rle)); ioa = np.asarray(ioa).reshape(len(d_rle), len(ig_rle))
        g_cls = np.array([a["category_id"] for a in G], int)
        cache[iid] = dict(iou=iou, g_cls=g_cls, d_idx=[k for k, _ in ds])
        for j, a in enumerate(G):
            G_rows.append(dict(image_id=iid, g=j, ann_id=a["id"], cls=a["category_id"], area=float(mask_util.area(g_rle[j]))))
        for q, (k, d) in enumerate(ds):
            same = iou[q, g_cls == d["category_id"]] if len(g_cls) else np.zeros(0)
            other = iou[q, g_cls != d["category_id"]] if len(g_cls) else np.zeros(0)
            D_rows.append(dict(image_id=iid, q=q, det=k, cls=d["category_id"], score=d["score"], area=d["area"],
                               max_iou_same=float(same.max()) if len(same) else 0.0, max_iou_other=float(other.max()) if len(other) else 0.0,
                               max_ioa_ignore=float(ioa[q].max()) if ioa.shape[1] else 0.0, bbox=d["bbox"]))
    return pd.DataFrame(D_rows), pd.DataFrame(G_rows), cache

def class_aware_match(D, Gt, cache, conf_th=None, iou_thr=0.5):
    # จับคู่แบบ COCO: แยกคลาส เรียงตาม score ภายในภาพ; คืนสถานะของแต่ละ detection และ GT
    D = D.copy(); Gt = Gt.copy()
    keep = np.ones(len(D), bool) if conf_th is None else D.apply(lambda r: r.score >= conf_th[CLASS_NAMES[r.cls - 1]], axis=1).values
    D = D[keep].reset_index(drop=True)
    D["kind"] = "FP"; D["gt"] = -1; D["iou"] = 0.0
    Gt["matched"] = False; Gt["iou"] = 0.0; Gt["det_score"] = np.nan
    gidx = {(r.image_id, r.g): i for i, r in Gt.iterrows()}
    for iid, dd in D.groupby("image_id"):
        c = cache[iid]; used = set()
        for i, r in dd.sort_values("score", ascending=False).iterrows():
            best, bj = iou_thr, -1
            for j in range(len(c["g_cls"])):
                if j in used or c["g_cls"][j] != r.cls:
                    continue
                v = c["iou"][r.q, j]
                if v >= best:
                    best, bj = v, j
            if bj >= 0:
                used.add(bj); D.loc[i, ["kind", "gt", "iou"]] = ["TP", bj, best]
                gi = gidx[(iid, bj)]; Gt.loc[gi, ["matched", "iou", "det_score"]] = [True, best, r.score]
            elif r.max_ioa_ignore >= 0.5:
                D.loc[i, "kind"] = "IGNORED"
    return D, Gt

def confusion_instance(D, Gt, cache, conf_th, iou_thr=0.5):
    labels = CLASS_NAMES + ["background"]
    C = np.zeros((3, 3), int)
    Dk = D[D.apply(lambda r: r.score >= conf_th[CLASS_NAMES[r.cls - 1]], axis=1)]
    for iid in cache:
        c = cache[iid]; dd = Dk[Dk.image_id == iid].sort_values("score", ascending=False)
        g_cls = c["g_cls"] if c else np.zeros(0, int); used = set()
        for _, r in dd.iterrows():
            if r.max_ioa_ignore >= 0.5 and max(r.max_iou_same, r.max_iou_other) < iou_thr:
                continue
            best, bj = iou_thr, -1
            for j in range(len(g_cls)):
                if j not in used and c["iou"][r.q, j] >= best:
                    best, bj = c["iou"][r.q, j], j
            if bj >= 0:
                used.add(bj); C[g_cls[bj] - 1, r.cls - 1] += 1
            else:
                C[2, r.cls - 1] += 1
        for j in range(len(g_cls)):
            if j not in used:
                C[g_cls[j] - 1, 2] += 1
    return pd.DataFrame(C, index=[f"GT {l}" for l in labels], columns=[f"pred {l}" for l in labels])

def plot_confusion(CM, title, path):
    M = CM.values.astype(float)
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.3))
    for ax, norm in zip(axes, (False, True)):
        A = M.copy()
        if norm:
            s = A.sum(1, keepdims=True); A = np.divide(A, s, out=np.zeros_like(A), where=s > 0)
        A_show = A.copy(); A_show[2, 2] = np.nan
        ax.imshow(np.ma.masked_invalid(A_show), cmap="Blues", vmin=0)
        for i in range(3):
            for j in range(3):
                if i == 2 and j == 2:
                    ax.text(j, i, "—", ha="center", va="center", color="#52514e"); continue
                v = A[i, j]; txt = f"{v:.2f}" if norm else f"{int(v)}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                        color="white" if v > np.nanmax(A_show) * 0.6 else "#0b0b0b")
        ax.set_xticks(range(3)); ax.set_xticklabels([c.replace("pred ", "") for c in CM.columns])
        ax.set_yticks(range(3)); ax.set_yticklabels([c.replace("GT ", "") for c in CM.index])
        ax.set_xlabel("predicted"); ax.set_ylabel("ground truth"); ax.grid(False)
        ax.set_title(("row-normalized" if norm else "counts"))
    fig.suptitle(title, fontsize=9); fig.tight_layout(); savefig(fig, path); plt.show()

D_VAL, G_VAL, CACHE_VAL = build_matches(GT_VAL, DETS_VAL, VAL)
DM_VAL, GM_VAL = class_aware_match(D_VAL, G_VAL, CACHE_VAL, CONF_TH)
CM_VAL = confusion_instance(D_VAL, G_VAL, CACHE_VAL, CONF_TH)
save_table(CM_VAL.reset_index(), TAB / "val_confusion_instance.csv")
plot_confusion(CM_VAL, f"Instance confusion matrix (val, mask IoU ≥ 0.5, conf {CONF_TH})", FIG / "fig04_val_confusion_instance")
summ = pd.DataFrame([dict(cls=c, TP=int(((DM_VAL.cls == i + 1) & (DM_VAL.kind == "TP")).sum()),
                          FP=int(((DM_VAL.cls == i + 1) & (DM_VAL.kind == "FP")).sum()),
                          FN=int(((GM_VAL.cls == i + 1) & ~GM_VAL.matched).sum()), GT=int((GM_VAL.cls == i + 1).sum()))
                     for i, c in enumerate(CLASS_NAMES)])
summ["precision"] = summ.TP / (summ.TP + summ.FP).clip(lower=1); summ["recall"] = summ.TP / summ.GT.clip(lower=1)
summ["F1"] = 2 * summ.TP / (2 * summ.TP + summ.FP + summ.FN).clip(lower=1)
save_table(summ, TAB / "val_tp_fp_fn_summary.csv"); display(summ.round(4))

## 6. การจำแนกประเภทความผิดพลาด (TIDE)

**ชุดข้อมูล:** val โดย ΔAP50 ใช้ detection ทุกตัว (confidence ≥ 0.001) และจำนวนความผิดพลาดนับที่ threshold ของหัวข้อ 3.1


ความผิดพลาดแต่ละรายการถูกจำแนกตามนิยามของ TIDE (Bolya et al., 2020) โดยใช้ foreground IoU 0.5 และ background IoU 0.1 กับ mask ดังนี้ Cls คือ detection ที่ซ้อนกับ GT ของคลาสอื่นอย่างน้อย 0.5, Loc คือ detection ที่ซ้อนกับ GT คลาสเดียวกันระหว่าง 0.1 ถึง 0.5, Both คือซ้อนกับ GT คลาสอื่นระหว่าง 0.1 ถึง 0.5, Dupe คือซ้อนกับ GT คลาสเดียวกันอย่างน้อย 0.5 แต่ GT นั้นถูกจับคู่ไปแล้ว, Bkg คือซ้อนกับ GT ทุกตัวน้อยกว่า 0.1 และ Miss คือ GT ที่ไม่ถูกตรวจพบและไม่ได้อธิบายด้วยความผิดพลาดประเภทอื่น

ผลกระทบต่อ AP (ΔAP50) คำนวณด้วยวิธี oracle ของ TIDE คือแก้ความผิดพลาดทีละประเภทแล้วคำนวณ AP50 ใหม่ ค่า ΔAP ที่สูงบอกว่าการแก้ความผิดพลาดประเภทนั้นจะเพิ่ม AP ได้มากที่สุด ค่าในหัวข้อนี้คำนวณด้วยโค้ดใน Notebook นี้ ไม่ได้ใช้ไลบรารี `tidecv` จึงอาจต่างจากไลบรารีต้นฉบับเล็กน้อยในรายละเอียดของการจัดลำดับการแก้

In [ ]:
ERR_TYPES = ["Cls", "Loc", "Both", "Dupe", "Bkg", "Miss"]

def tide_assign(D, Gt, cache, fg=0.5, bg=0.1):
    # จับคู่ทุก detection (ไม่ตัด threshold) แบบ COCO แล้วจำแนกประเภทของ FP และ FN
    D = D.sort_values("score", ascending=False).reset_index(drop=True).copy()
    D["kind"] = ""; D["target"] = -1; D["target_cls"] = -1
    matched = set(); gkey = lambda iid, j: (iid, j)
    for i, r in D.iterrows():
        c = cache[r.image_id]; g_cls = c["g_cls"]; iou = c["iou"][r.q] if len(g_cls) else np.zeros(0)
        cand = [(iou[j], j) for j in range(len(g_cls)) if g_cls[j] == r.cls and gkey(r.image_id, j) not in matched and iou[j] >= fg]
        if cand:
            v, j = max(cand); matched.add(gkey(r.image_id, j)); D.loc[i, ["kind", "target", "target_cls"]] = ["TP", j, r.cls]; continue
        if r.max_ioa_ignore >= 0.5 and max(r.max_iou_same, r.max_iou_other) < fg:
            D.loc[i, "kind"] = "IGN"; continue
        same = [(iou[j], j) for j in range(len(g_cls)) if g_cls[j] == r.cls]
        other = [(iou[j], j) for j in range(len(g_cls)) if g_cls[j] != r.cls]
        ms, js = max(same) if same else (0, -1); mo, jo = max(other) if other else (0, -1)
        if ms >= fg:
            k, j = "Dupe", js
        elif mo >= fg:
            k, j = "Cls", jo
        elif ms >= bg:
            k, j = "Loc", js
        elif mo >= bg:
            k, j = "Both", jo
        else:
            k, j = "Bkg", -1
        D.loc[i, ["kind", "target", "target_cls"]] = [k, j, int(g_cls[j]) if j >= 0 else -1]
    Gt = Gt.copy(); Gt["detected"] = [(r.image_id, r.g) in matched for r in Gt.itertuples()]
    explained = set((r.image_id, r.target) for r in D[D.kind.isin(["Cls", "Loc", "Both"])].itertuples())
    Gt["err"] = np.where(Gt.detected, "", np.where([(r.image_id, r.g) in explained for r in Gt.itertuples()], "explained", "Miss"))
    return D, Gt

def ap50(scores, tp, npos):
    if npos == 0:
        return float("nan")
    if len(scores) == 0:
        return 0.0
    o = np.argsort(-np.asarray(scores), kind="mergesort"); tp = np.asarray(tp, bool)[o]
    ctp = np.cumsum(tp); cfp = np.cumsum(~tp)
    rc = ctp / npos; pr = ctp / np.maximum(ctp + cfp, 1e-12)
    pr = np.maximum.accumulate(pr[::-1])[::-1]
    idx = np.searchsorted(rc, np.linspace(0, 1, 101), side="left")
    return float(np.where(idx < len(pr), pr[np.minimum(idx, len(pr) - 1)], 0.0).mean())

def tide_delta(DT, GT):
    def mean_ap(fix=None):
        aps = []
        for c in (1, 2):
            d = DT[DT.kind != "IGN"].copy(); g = GT[GT.cls == c]
            npos = len(g); claimed = set(zip(d[d.kind == "TP"].image_id, d[d.kind == "TP"].target))
            rows = []
            for r in d.sort_values("score", ascending=False).itertuples():
                k = r.kind
                if fix == k and k in ("Dupe", "Bkg", "Both"):
                    continue
                if fix == k and k in ("Cls", "Loc"):
                    tc = r.target_cls if k == "Cls" else r.cls
                    if tc == c and (r.image_id, r.target) not in claimed:
                        claimed.add((r.image_id, r.target)); rows.append((r.score, True))
                    continue
                if r.cls == c:
                    rows.append((r.score, k == "TP"))
            if fix == "Miss":
                npos -= int(((GT.cls == c) & (GT.err == "Miss")).sum())
            aps.append(ap50([s for s, _ in rows], [t for _, t in rows], npos))
        return float(np.nanmean(aps))
    base = mean_ap()
    return base, {k: mean_ap(k) - base for k in ERR_TYPES}

def tide_report(D, Gt, cache, conf_th, tag):
    DT, GT = tide_assign(D, Gt, cache)
    base, delta = tide_delta(DT, GT)
    keep = DT.apply(lambda r: r.score >= conf_th[CLASS_NAMES[r.cls - 1]], axis=1)
    DTk = DT[keep]
    _, GTk = tide_assign(D[D.apply(lambda r: r.score >= conf_th[CLASS_NAMES[r.cls - 1]], axis=1)], Gt, cache)
    cnt = {k: int((DTk.kind == k).sum()) for k in ERR_TYPES[:-1]}; cnt["Miss"] = int((GTk.err == "Miss").sum())
    R = pd.DataFrame([dict(error=k, delta_AP50=delta[k], count_at_conf_th=cnt[k]) for k in ERR_TYPES])
    save_table(R, TAB / f"{tag}_tide_errors.csv")
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 2.8))
    cols = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
    axes[0].barh(R.error, R.delta_AP50 * 100, color=cols, height=0.6)
    for y, v in enumerate(R.delta_AP50 * 100):
        axes[0].text(v, y, f" {v:.2f}", va="center", fontsize=7)
    axes[0].invert_yaxis(); axes[0].set_xlabel("ΔAP50 if fixed (points)"); axes[0].set_title(f"TIDE-style error impact (base mask AP50 {base*100:.1f})")
    axes[1].barh(R.error, R.count_at_conf_th, color=cols, height=0.6)
    for y, v in enumerate(R.count_at_conf_th):
        axes[1].text(v, y, f" {v}", va="center", fontsize=7)
    axes[1].invert_yaxis(); axes[1].set_xlabel("count at operating threshold"); axes[1].set_title("error counts")
    fig.tight_layout(); savefig(fig, FIG / f"fig05_{tag}_tide_errors"); plt.show()
    display(R.round(4))
    return DT, GT, DTk, GTk

DT_VAL, GT_TIDE_VAL, DTK_VAL, GTK_VAL = tide_report(D_VAL, G_VAL, CACHE_VAL, CONF_TH, "val")

## 7. คุณภาพของ mask

**ชุดข้อมูล:** TP ของชุด val ที่ threshold ของหัวข้อ 3.1


สำหรับทุก TP ที่ threshold ที่ใช้งาน คำนวณ mask IoU และ Boundary IoU (Cheng et al., 2021) โดยใช้ dilation ratio 0.02 ของเส้นทแยงมุมของภาพ Boundary IoU วัดความถูกต้องเฉพาะแถบรอบขอบของวัตถุ จึงไวต่อความคลาดเคลื่อนของขอบมากกว่า mask IoU ซึ่งถูกครอบงำด้วยพื้นที่ภายในของวัตถุขนาดใหญ่

In [ ]:
def boundary_iou(gt_mask, dt_mask, dilation_ratio=0.02):
    h, w = gt_mask.shape; d = max(1, int(round(dilation_ratio * np.hypot(h, w))))
    k = np.ones((3, 3), np.uint8)
    def band(m):
        return m.astype(bool) & ~cv2.erode(m.astype(np.uint8), k, iterations=d).astype(bool)
    g, p = band(gt_mask), band(dt_mask); u = (g | p).sum()
    return float((g & p).sum() / u) if u else float("nan")

def mask_quality(DM, dets, gt_json, tag):
    gt = load_gt(gt_json); rows = []
    for r in DM[DM.kind == "TP"].itertuples():
        anns = [a for a in gt.imgToAnns[r.image_id] if not a.get("iscrowd", 0)]
        gm = gt.annToMask(anns[r.gt]); dm = mask_util.decode(dets[r.det]["segmentation"])
        rows.append(dict(image_id=r.image_id, cls=CLASS_NAMES[r.cls - 1], score=r.score, gt_area=float(gm.sum()),
                         mask_iou=r.iou, boundary_iou=boundary_iou(gm, dm)))
    Q = pd.DataFrame(rows); save_table(Q, TAB / f"{tag}_mask_quality.csv")
    fig, axes = plt.subplots(1, 3, figsize=(11, 2.9))
    for c in CLASS_NAMES:
        q = Q[Q.cls == c]
        axes[0].hist(q.mask_iou, bins=np.linspace(0.5, 1, 26), alpha=0.6, color=CLASS_COLORS[c], label=f"{c} (median {q.mask_iou.median():.3f})")
        axes[1].hist(q.boundary_iou, bins=np.linspace(0, 1, 26), alpha=0.6, color=CLASS_COLORS[c], label=f"{c} (median {q.boundary_iou.median():.3f})")
        axes[2].scatter(q.mask_iou, q.boundary_iou, s=10, alpha=0.6, color=CLASS_COLORS[c], label=c, edgecolors="none")
    axes[0].set_xlabel("mask IoU"); axes[0].set_title("mask IoU of TPs"); axes[0].legend(frameon=False, fontsize=6.5)
    axes[1].set_xlabel("Boundary IoU"); axes[1].set_title("Boundary IoU of TPs"); axes[1].legend(frameon=False, fontsize=6.5)
    axes[2].plot([0, 1], [0, 1], color="#52514e", lw=0.8, ls=":"); axes[2].set_xlim(0.45, 1); axes[2].set_ylim(0, 1)
    axes[2].set_xlabel("mask IoU"); axes[2].set_ylabel("Boundary IoU"); axes[2].set_title("interior vs boundary quality"); axes[2].legend(frameon=False, fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / f"fig06_{tag}_mask_quality"); plt.show()
    display(Q.groupby("cls")[["mask_iou", "boundary_iou"]].describe().round(3))
    return Q

Q_VAL = mask_quality(DM_VAL, DETS_VAL, GT_VAL, "val")

## 8. ผลตามขนาดของวัตถุ

**ชุดข้อมูล:** GT และ detection ของชุด val ที่ threshold ของหัวข้อ 3.1


นอกจาก AP ตามช่วงขนาดมาตรฐานของ COCO (small < 32², medium < 96², large ≥ 96² พิกเซล) ซึ่งรายงานในหัวข้อ 3 แล้ว ส่วนนี้แบ่ง GT ออกเป็นช่วงตามควอนไทล์ของพื้นที่ในชุดข้อมูลนี้ เพราะวัตถุส่วนใหญ่ในภาพของกองวัสดุมีขนาดใหญ่ ทำให้ช่วงมาตรฐานของ COCO มีจำนวนตัวอย่างไม่สมดุล

In [ ]:
def size_analysis(GM, DM, tag, n_bins=5):
    G = GM.copy(); G["area_frac"] = G.area / float(VAL.width.iloc[0] * VAL.height.iloc[0])
    edges = np.unique(np.quantile(G.area, np.linspace(0, 1, n_bins + 1)))
    G["bin"] = pd.cut(G.area, edges, include_lowest=True)
    R = G.groupby(["bin", "cls"], observed=True).agg(n=("matched", "size"), recall=("matched", "mean"), mean_iou=("iou", "mean")).reset_index()
    R["cls"] = R.cls.map(lambda c: CLASS_NAMES[c - 1]); R["bin"] = R.bin.astype(str)
    save_table(R, TAB / f"{tag}_recall_by_size.csv")
    D = DM[DM.kind.isin(["TP", "FP"])].copy()
    D["bin"] = pd.cut(D.area.clip(edges[0], edges[-1]), edges, include_lowest=True).astype(str)
    P = D.groupby("bin").kind.apply(lambda s: (s == "TP").mean()).rename("precision").reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 2.9))
    bins = [str(b) for b in pd.cut(G.area, edges, include_lowest=True).cat.categories]
    x = np.arange(len(bins)); w = 0.38
    for k, c in enumerate(CLASS_NAMES):
        r = R[R.cls == c].set_index("bin").reindex(bins)
        axes[0].bar(x + (k - 0.5) * w, r.recall, w * 0.92, color=CLASS_COLORS[c], label=c)
        for xi, (rv, n) in enumerate(zip(r.recall, r.n)):
            if not np.isnan(rv):
                axes[0].text(xi + (k - 0.5) * w, rv, f"{int(n)}", ha="center", va="bottom", fontsize=5.5, color="#52514e")
    lab = [f"{int(float(b.split(',')[0].strip('(['))):,}–\n{int(float(b.split(',')[1].strip(')]'))):,}" for b in bins]
    axes[0].set_xticks(x); axes[0].set_xticklabels(lab, fontsize=6); axes[0].set_ylim(0, 1.08)
    axes[0].set_xlabel("GT area (px), quantile bins"); axes[0].set_ylabel("recall"); axes[0].set_title("recall by object size (n on bars)")
    axes[0].legend(frameon=False, fontsize=6.5)
    p = P.set_index("bin").reindex(bins)
    axes[1].bar(x, p.precision, 0.6, color="#2a78d6"); axes[1].set_xticks(x); axes[1].set_xticklabels(lab, fontsize=6)
    axes[1].set_ylim(0, 1.05); axes[1].set_xlabel("predicted area (px)"); axes[1].set_ylabel("precision"); axes[1].set_title("precision by predicted size")
    for c in CLASS_NAMES:
        g = G[G.cls == CLASS_NAMES.index(c) + 1]
        axes[2].scatter(g.area, g.iou, s=9, alpha=0.6, color=CLASS_COLORS[c], label=c, edgecolors="none")
    axes[2].axhline(0.5, color="#52514e", ls=":", lw=0.8); axes[2].set_xscale("log")
    axes[2].set_xlabel("GT area (px, log)"); axes[2].set_ylabel("IoU of matched detection (0 = missed)")
    axes[2].set_title("per-GT IoU vs size"); axes[2].legend(frameon=False, fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / f"fig07_{tag}_size_analysis"); plt.show()
    display(R.round(3))

size_analysis(GM_VAL, DM_VAL, "val")

## 9. ความสอดคล้องของ confidence (calibration)

**ชุดข้อมูล:** detection ของชุด val ที่ confidence ≥ 0.05


แบบจำลองที่ calibrate ดีควรมีสัดส่วนของ detection ที่ถูกต้องใกล้เคียงกับค่า confidence ที่ให้ไว้ ส่วนนี้ใช้ detection ทุกตัวที่มี confidence อย่างน้อย 0.05 แบ่งเป็น 15 ช่วงเท่ากัน (ตาม Küppers et al., 2020) และนิยามว่า detection ถูกต้องเมื่อเป็น TP ที่ mask IoU ≥ 0.5 แล้วคำนวณ Expected Calibration Error (D-ECE แบบใช้ confidence อย่างเดียว) เป็นค่าเฉลี่ยถ่วงน้ำหนักของ |precision − confidence| ในแต่ละช่วง

In [ ]:
def calibration(D, Gt, cache, tag, n_bins=15, min_conf=0.05):
    DM, _ = class_aware_match(D[D.score >= min_conf], Gt, cache, None)
    DM = DM[DM.kind != "IGNORED"]
    edges = np.linspace(min_conf, 1, n_bins + 1)
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.0), gridspec_kw=dict(width_ratios=[1.3, 1]))
    rows = []
    for c in CLASS_NAMES:
        d = DM[DM.cls == CLASS_NAMES.index(c) + 1]
        b = np.clip(np.digitize(d.score, edges) - 1, 0, n_bins - 1)
        g = pd.DataFrame(dict(b=b, conf=d.score.values, ok=(d.kind == "TP").values)).groupby("b").agg(conf=("conf", "mean"), acc=("ok", "mean"), n=("ok", "size"))
        ece = float((g.n / g.n.sum() * (g.acc - g.conf).abs()).sum())
        rows.append(dict(cls=c, D_ECE=ece, n_dets=len(d)))
        axes[0].plot(g.conf, g.acc, "-o", ms=4, lw=2, color=CLASS_COLORS[c], label=f"{c} (D-ECE {ece:.3f})")
        axes[1].hist(d.score, bins=edges, alpha=0.6, color=CLASS_COLORS[c], label=c)
    axes[0].plot([0, 1], [0, 1], color="#52514e", lw=0.8, ls=":", label="perfect calibration")
    axes[0].set_xlabel("mean confidence in bin"); axes[0].set_ylabel("precision in bin (IoU ≥ 0.5)")
    axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1); axes[0].set_title("reliability diagram"); axes[0].legend(frameon=False, fontsize=6.5)
    axes[1].set_yscale("log"); axes[1].set_xlabel("confidence"); axes[1].set_ylabel("detections (log)"); axes[1].set_title("confidence distribution")
    axes[1].legend(frameon=False, fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / f"fig08_{tag}_calibration"); plt.show()
    R = pd.DataFrame(rows); save_table(R, TAB / f"{tag}_calibration.csv"); display(R.round(4))

calibration(D_VAL, G_VAL, CACHE_VAL, "val")

## 10. ผลรายกลุ่มของฉากและรายสถานะ

**ชุดข้อมูล:** val แยกตาม `group_merged` และสถานะของภาพ


ชุด validation ประกอบด้วยหลายกลุ่มของฉาก (`group_merged` ซึ่งรวมภาพจากวิดีโอหรือเหตุการณ์เดียวกัน) ผลรายกลุ่มแสดงว่าความผิดพลาดกระจุกตัวอยู่ในบางฉากหรือไม่ จำนวนภาพในแต่ละกลุ่มแสดงไว้ในกราฟ เนื่องจากกลุ่มที่มีภาพน้อยมีความแปรปรวนสูง

In [ ]:
def per_group(coco, DM, GM, records, tag):
    recs = coco["segm"]["recs"]; rows = []
    for g, d in records.groupby("group_merged"):
        ii = d.image_id.astype(int).tolist()
        dm = DM[DM.image_id.isin(ii)]; gm = GM[GM.image_id.isin(ii)]
        tp = int((dm.kind == "TP").sum()); fp = int((dm.kind == "FP").sum()); fn = int((~gm.matched).sum())
        rows.append(dict(group=g[:8], n_images=len(ii), state=d.state.mode()[0], mask_AP_gravel=ap_from_records(recs, ii, 1),
                         mask_AP_sand=ap_from_records(recs, ii, 2), TP=tp, FP=fp, FN=fn, F1=2 * tp / max(2 * tp + fp + fn, 1)))
    P = pd.DataFrame(rows).sort_values("F1"); save_table(P, TAB / f"{tag}_per_group.csv")
    fig, axes = plt.subplots(1, 2, figsize=(11, max(2.6, 0.24 * len(P) + 0.8)))
    y = np.arange(len(P))
    axes[0].barh(y, P.F1, color=[CLASS_COLORS.get(s, "#999999") for s in P.state], height=0.65)
    axes[0].set_yticks(y); axes[0].set_yticklabels([f"{g} (n={n})" for g, n in zip(P.group, P.n_images)], fontsize=6.5)
    axes[0].set_xlim(0, 1); axes[0].set_xlabel("instance F1 at operating threshold"); axes[0].set_title("per-scene F1 (colour = scene state)")
    axes[0].legend(handles=[Rectangle((0, 0), 1, 1, color=CLASS_COLORS[s], label=s) for s in STATES if s in set(P.state)], frameon=False, fontsize=6.5, loc="lower right")
    axes[1].barh(y, P.FP, color=KIND_COLORS["FP"], height=0.65, label="FP")
    axes[1].barh(y, P.FN, left=P.FP, color=KIND_COLORS["FN"], height=0.65, label="FN")
    axes[1].set_yticks(y); axes[1].set_yticklabels([]); axes[1].set_xlabel("errors (count)"); axes[1].set_title("FP and FN per scene")
    axes[1].legend(frameon=False, fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / f"fig09_{tag}_per_group"); plt.show()
    S = []
    for s, d in records.groupby("state"):
        ii = d.image_id.astype(int).tolist(); dm = DM[DM.image_id.isin(ii)]; gm = GM[GM.image_id.isin(ii)]
        S.append(dict(state=s, n_images=len(ii), TP=int((dm.kind == "TP").sum()), FP=int((dm.kind == "FP").sum()), FN=int((~gm.matched).sum()),
                      mask_AP_gravel=ap_from_records(recs, ii, 1), mask_AP_sand=ap_from_records(recs, ii, 2)))
    S = pd.DataFrame(S); save_table(S, TAB / f"{tag}_per_state.csv"); display(S.round(4))
    return P

PG_VAL = per_group(COCO_VAL, DM_VAL, GM_VAL, VAL, "val")

## 11. การจำแนกสถานะระดับภาพ (L2)

**ชุดข้อมูล:** val ทั้ง 131 ภาพ เกณฑ์ $\theta_c$ ถูกกำหนดจากชุดเดียวกันนี้ใน Notebook 03 ผลจึงเป็นขอบเขตบนของประสิทธิภาพ


สถานะของภาพ (none, gravel, sand, mixed) ได้จากคะแนน $S_c = \sum \text{conf} \times \text{พื้นที่ mask} / \text{พื้นที่ภาพ}$ ของแต่ละคลาส เทียบกับเกณฑ์ $\theta_c$ ซึ่งเป็นวิธีเดียวกับ Notebook 02 และ 03 กราฟซ้ายแสดง confusion matrix ของสถานะ และกราฟขวาแสดงคะแนนของทุกภาพพร้อมเส้นเกณฑ์ ซึ่งช่วยให้เห็นว่าภาพที่จำแนกผิดอยู่ใกล้เกณฑ์หรืออยู่ห่างจากเกณฑ์

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

def state_scores(dets, records):
    area = dict(zip(records.image_id.astype(int), records.width * records.height)); S = defaultdict(lambda: np.zeros(2))
    for d in dets:
        if d["score"] >= STATE_MIN_CONF and d["image_id"] in area:
            S[d["image_id"]][d["category_id"] - 1] += d["score"] * d["area"] / area[d["image_id"]]
    return pd.DataFrame([dict(image_id=i, S_gravel=S[i][0], S_sand=S[i][1]) for i in records.image_id.astype(int)])

def fit_state_thresholds(scores, records):
    df = scores.merge(records[["image_id", "state"]], on="image_id"); out = {}
    for c in CLASS_NAMES:
        y = df.state.isin([c, "mixed"]).values; x = df[f"S_{c}"].values
        cands = np.unique(np.concatenate([[1e-6], np.quantile(x, np.linspace(0, 1, 201))])); f1 = []
        for th in cands:
            p = x >= th; tp = (p & y).sum(); fp = (p & ~y).sum(); fn = (~p & y).sum(); f1.append(2 * tp / max(2 * tp + fp + fn, 1))
        out[c] = float(cands[int(np.argmax(f1))])
    return out

def state_eval(dets, records, th, tag):
    sc = state_scores(dets, records).merge(records[["image_id", "state", "group_merged"]], on="image_id")
    g = sc.S_gravel.values >= th["gravel"]; s = sc.S_sand.values >= th["sand"]
    sc["pred"] = np.where(g & s, "mixed", np.where(g, "gravel", np.where(s, "sand", "none")))
    labels = [x for x in STATES if x in set(sc.state) | set(sc.pred)]
    cm = confusion_matrix(sc.state, sc.pred, labels=labels)
    res = dict(n=len(sc), accuracy=float((sc.state == sc.pred).mean()), balanced_accuracy=float(balanced_accuracy_score(sc.state, sc.pred)),
               macro_f1=float(f1_score(sc.state, sc.pred, average="macro", labels=labels, zero_division=0)),
               kappa=float(cohen_kappa_score(sc.state, sc.pred)), thresholds=th)
    save_table(sc, TAB / f"{tag}_state_predictions.csv"); (TAB / f"{tag}_state_metrics.json").write_text(json.dumps(res, indent=2))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    ax = axes[0]; ax.imshow(cm, cmap="Blues")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() * 0.6 else "#0b0b0b")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels); ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("predicted state"); ax.set_ylabel("true state"); ax.grid(False)
    ax.set_title(f"state confusion (macro-F1 {res['macro_f1']:.3f}, bal.acc {res['balanced_accuracy']:.3f})")
    ax = axes[1]; eps = 1e-4
    for st in labels:
        d = sc[sc.state == st]; wrong = d.pred != d.state
        ax.scatter(d.S_gravel[~wrong] + eps, d.S_sand[~wrong] + eps, s=14, color=CLASS_COLORS[st], label=st, edgecolors="none", alpha=0.8)
        ax.scatter(d.S_gravel[wrong] + eps, d.S_sand[wrong] + eps, s=30, facecolors="none", edgecolors=CLASS_COLORS[st], linewidths=1.3)
    ax.axvline(th["gravel"] + eps, color="#52514e", ls=":", lw=1); ax.axhline(th["sand"] + eps, color="#52514e", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("S_gravel (log)"); ax.set_ylabel("S_sand (log)")
    ax.set_title("image scores (open circle = misclassified)"); ax.legend(frameon=False, fontsize=6.5, title="true state", title_fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / f"fig10_{tag}_state_classification"); plt.show()
    print(json.dumps(res, indent=2))
    return sc

TH_STATE = FROZEN["state_thresholds"] if FROZEN is not None else fit_state_thresholds(state_scores(DETS_VAL, VAL), VAL)
SC_VAL = state_eval(DETS_VAL, VAL, TH_STATE, "val")

## 12. ภาพตัวอย่าง TP / FP / FN

**ชุดข้อมูล:** val


ภาพในส่วนนี้ใช้ threshold ที่ใช้งาน สัญลักษณ์มีดังนี้ เส้นประสีขาวคือขอบของ GT, พื้นที่สีเขียวคือ TP, พื้นที่สีแดงคือ FP และเส้นทึบสีเหลืองคือขอบของ GT ที่ไม่ถูกตรวจพบ (FN) ภาพแรกแสดงภาพที่มีความผิดพลาดมากที่สุดและน้อยที่สุด ส่วนภาพถัดไปแสดง crop ของ FP ที่มี confidence สูงที่สุดพร้อมประเภทตาม TIDE และ crop ของ FN ที่มีขนาดใหญ่ที่สุด

In [ ]:
def contour(ax, mask, color, ls="-", lw=1.0):
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in cs:
        if len(c) >= 3:
            xy = c[:, 0, :]; ax.plot(np.r_[xy[:, 0], xy[0, 0]], np.r_[xy[:, 1], xy[0, 1]], color=color, ls=ls, lw=lw)

def fill(ax, mask, color, alpha=0.42):
    rgba = np.zeros((*mask.shape, 4)); rgba[mask > 0] = [*matplotlib.colors.to_rgb(color), alpha]; ax.imshow(rgba)

def draw_image(ax, rec, DM, GM, dets, gt, title=None, transform=None):
    img = cv2.imread(rec.path); img = transform(img) if transform else img
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    anns = [a for a in gt.imgToAnns[int(rec.image_id)] if not a.get("iscrowd", 0)]
    for a in gt.imgToAnns[int(rec.image_id)]:
        if a.get("iscrowd", 0):
            fill(ax, gt.annToMask(a), "#999999", 0.35)
    dm = DM[DM.image_id == rec.image_id]
    for r in dm.itertuples():
        m = mask_util.decode(dets[r.det]["segmentation"])
        if r.kind == "TP":
            fill(ax, m, KIND_COLORS["TP"])
        elif r.kind == "FP":
            fill(ax, m, KIND_COLORS["FP"]); x, y, w, h = r.bbox
            ax.text(x, y, f"{CLASS_NAMES[r.cls-1][0]} {r.score:.2f}", fontsize=5.5, color="white", va="bottom",
                    bbox=dict(boxstyle="square,pad=0.1", fc=KIND_COLORS["FP"], ec="none"))
    gm = GM[GM.image_id == rec.image_id]
    for r in gm.itertuples():
        m = gt.annToMask(anns[r.g])
        contour(ax, m, "white", "--", 0.9)
        if not r.matched:
            contour(ax, m, KIND_COLORS["FN"], "-", 1.6)
    if title:
        ax.set_title(title, fontsize=6.5)

def legend_handles():
    return [Rectangle((0, 0), 1, 1, color=KIND_COLORS["TP"], alpha=0.6, label="TP"), Rectangle((0, 0), 1, 1, color=KIND_COLORS["FP"], alpha=0.6, label="FP"),
            Line2D([], [], color=KIND_COLORS["FN"], lw=2, label="FN (missed GT)"), Line2D([], [], color="#52514e", ls="--", lw=1, label="GT outline (white dashed)"),
            Rectangle((0, 0), 1, 1, color="#999999", alpha=0.5, label="ignore region")]

def diverse(ids, records, n, per_group=2, min_gap=15):
    # เลือกภาพที่ไม่ซ้ำกันมาก: ไม่เกิน per_group ภาพต่อกลุ่ม และเฟรมของวิดีโอเดียวกันห่างกันอย่างน้อย min_gap
    meta = records.set_index("image_id"); out, cnt = [], Counter()
    for i in ids:
        g, f = meta.loc[i, "group_merged"], meta.loc[i, "frame"]
        if cnt[g] >= per_group or any(meta.loc[j, "group_merged"] == g and abs(meta.loc[j, "frame"] - f) < min_gap for j in out):
            continue
        out.append(i); cnt[g] += 1
        if len(out) == n:
            break
    return out

def diverse_rows(df, records, n, per_image_group=4, min_gap=8):
    meta = records.set_index("image_id"); keep, cnt = [], Counter()
    for idx, r in df.iterrows():
        g, f = meta.loc[r.image_id, "group_merged"], meta.loc[r.image_id, "frame"]
        if cnt[g] >= per_image_group or any(meta.loc[df.loc[k, "image_id"], "group_merged"] == g and
                                            abs(meta.loc[df.loc[k, "image_id"], "frame"] - f) < min_gap for k in keep):
            continue
        keep.append(idx); cnt[g] += 1
        if len(keep) == n:
            break
    return df.loc[keep]

def galleries(records, DM, GM, DTK, dets, gt_json, tag, n=6):
    gt = load_gt(gt_json)
    err = (DM[DM.kind == "FP"].groupby("image_id").size().reindex(records.image_id, fill_value=0)
           + GM[~GM.matched].groupby("image_id").size().reindex(records.image_id, fill_value=0))
    tp = DM[DM.kind == "TP"].groupby("image_id").size().reindex(records.image_id, fill_value=0)
    for name, ids in (("worst", diverse(err.sort_values(ascending=False).index, records, n)),
                      ("best", diverse((err - 0.001 * tp).sort_values().index, records, n))):
        fig, axes = plt.subplots(1, n, figsize=(2.3 * n, 4.2))
        for ax, iid in zip(axes, ids):
            rec = records[records.image_id == iid].iloc[0]
            e = DM[DM.image_id == iid].kind.value_counts().to_dict(); fn = int((~GM[GM.image_id == iid].matched).sum())
            draw_image(ax, rec, DM, GM, dets, gt, f"{rec.group_merged[:8]} [{rec.state}]\nTP {e.get('TP',0)} FP {e.get('FP',0)} FN {fn}")
        fig.legend(handles=legend_handles(), loc="lower center", ncol=5, frameon=False, fontsize=7, bbox_to_anchor=(0.5, -0.02))
        fig.suptitle(f"{tag}: {name} images by FP+FN at operating threshold (at most 2 per scene, frames ≥ 15 apart)", fontsize=9)
        fig.tight_layout(rect=(0, 0.04, 1, 0.94)); savefig(fig, FIG / f"fig11_{tag}_{name}_images"); plt.show()
    # crop ของ FP ที่ confidence สูงที่สุด พร้อมประเภทตาม TIDE
    tide_kind = {(r.image_id, r.det): r.kind for r in DTK.itertuples()}
    fps = diverse_rows(DM[DM.kind == "FP"].sort_values("score", ascending=False), records, 12)
    def crop_panel(ax, rec, box, pad=0.25):
        img = cv2.cvtColor(cv2.imread(rec.path), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
        x, y, w, h = box; px, py = w * pad + 10, h * pad + 10
        x0, y0, x1, y1 = int(max(0, x - px)), int(max(0, y - py)), int(min(W, x + w + px)), int(min(H, y + h + py))
        return img, (x0, y0, x1, y1)
    if len(fps):
        cols = 6; rows_ = int(np.ceil(len(fps) / cols))
        fig, axes = plt.subplots(rows_, cols, figsize=(2.1 * cols, 2.5 * rows_)); axes = np.atleast_1d(axes).ravel()
        for ax, r in zip(axes, fps.itertuples()):
            rec = records[records.image_id == r.image_id].iloc[0]; img, (x0, y0, x1, y1) = crop_panel(ax, rec, r.bbox)
            ax.imshow(img[y0:y1, x0:x1]); m = mask_util.decode(dets[r.det]["segmentation"])[y0:y1, x0:x1]
            fill(ax, m, KIND_COLORS["FP"], 0.4)
            anns = [a for a in gt.imgToAnns[int(r.image_id)] if not a.get("iscrowd", 0)]
            for a in anns:
                contour(ax, gt.annToMask(a)[y0:y1, x0:x1], "white", "--", 0.9)
            ax.set_title(f"pred {CLASS_NAMES[r.cls-1]} {r.score:.2f}\nTIDE: {tide_kind.get((r.image_id, r.det), '?')}", fontsize=6.5)
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for ax in axes[len(fps):]:
            ax.axis("off")
        fig.suptitle(f"{tag}: highest-confidence false positives (red = prediction, white dashed = GT)", fontsize=9)
        fig.tight_layout(); savefig(fig, FIG / f"fig12_{tag}_fp_gallery"); plt.show()
    # crop ของ FN ที่ใหญ่ที่สุด พร้อม IoU สูงสุดที่ detection ใด ๆ ทำได้
    fns = diverse_rows(GM[~GM.matched].sort_values("area", ascending=False), records, 12)
    if len(fns):
        cols = 6; rows_ = int(np.ceil(len(fns) / cols))
        fig, axes = plt.subplots(rows_, cols, figsize=(2.1 * cols, 2.5 * rows_)); axes = np.atleast_1d(axes).ravel()
        for ax, r in zip(axes, fns.itertuples()):
            rec = records[records.image_id == r.image_id].iloc[0]
            anns = [a for a in gt.imgToAnns[int(r.image_id)] if not a.get("iscrowd", 0)]; a = anns[r.g]
            img, (x0, y0, x1, y1) = crop_panel(ax, rec, a["bbox"])
            ax.imshow(img[y0:y1, x0:x1]); contour(ax, gt.annToMask(a)[y0:y1, x0:x1], KIND_COLORS["FN"], "-", 1.6)
            c = CACHE_ALL[tag][r.image_id]; best = float(c["iou"][:, r.g].max()) if c["iou"].shape[0] else 0.0
            ax.set_title(f"GT {CLASS_NAMES[r.cls-1]} area {int(r.area):,}\nbest IoU any det {best:.2f}", fontsize=6.5)
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for ax in axes[len(fns):]:
            ax.axis("off")
        fig.suptitle(f"{tag}: largest missed ground-truth instances (yellow = GT outline)", fontsize=9)
        fig.tight_layout(); savefig(fig, FIG / f"fig13_{tag}_fn_gallery"); plt.show()

CACHE_ALL = {"val": CACHE_VAL}
galleries(VAL, DM_VAL, GM_VAL, DT_VAL, DETS_VAL, GT_VAL, "val")

## 13. ความทนทานต่อสภาพแวดล้อม (val-C)

**ชุดข้อมูล:** val-C คือภาพทั้ง 131 ภาพของ val ที่ Notebook 02 ใส่การรบกวนจำลอง 3 สถานการณ์ (dark, dust, rain) × 5 ระดับ รวม 15 ชุด โดยใช้ GT ของ val


val-C คือชุด validation ที่สร้างการรบกวนจำลองตามสถานการณ์ของหน้างาน (เช่น แสงน้อย ฝุ่น และฝน) ที่ระดับความรุนแรง 1 ถึง 5 ใน Notebook 02 ตัวชี้วัดคือ mPC (ค่าเฉลี่ยของ mask AP บนทุกการรบกวนและทุกระดับ) และ rPC = mPC / AP บนภาพปกติ (Hendrycks & Dietterich, 2019; Michaelis et al., 2019)

In [ ]:
VAL_C_DIRS = sorted((ROOT / "data").glob("val_c_*"))
if RUN_VALC and VAL_C_DIRS:
    VAL_C_INDEX = json.loads((VAL_C_DIRS[-1] / "index.json").read_text())
    rows = []
    for name, d in VAL_C_INDEX.items():
        recs = VAL.copy(); recs["path"] = [str(Path(d) / "images" / n) for n in recs.silver_name]
        dc = cached_dets(EVAL / f"dets_valC_{name}_{EVAL_TAG}.json.gz", lambda: predict_dets(WEIGHTS, recs, imgsz=IMGSZ, transform=TF, desc=name))
        E = run_cocoeval(load_gt(GT_VAL), dc, "segm", ids_val)
        rows.append(dict(set=name, scenario=name.split("_s")[0], level=int(name.split("_s")[1]), mask_AP=float(E.stats[0]), mask_AP50=float(E.stats[1])))
    VC = pd.DataFrame(rows); save_table(VC, TAB / "valC_mask_ap.csv")
    clean = T_VAL[(T_VAL.type == "mask") & (T_VAL.protocol == "ignore") & (T_VAL.metric == "AP")].value.iloc[0]
    mpc = VC.mask_AP.mean()
    print(f"clean mask AP {clean:.4f} | mPC {mpc:.4f} | rPC {mpc/clean:.3f}")
    scen = sorted(VC.scenario.unique()); cols = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#008300", "#e34948"]
    fig, ax = plt.subplots(figsize=(5.4, 3.1))
    for k, s in enumerate(scen):
        v = VC[VC.scenario == s].sort_values("level")
        ax.plot(np.r_[0, v.level], np.r_[clean, v.mask_AP], "-o", ms=4, lw=2, color=cols[k % len(cols)], label=f"{s} (mean {v.mask_AP.mean():.3f})")
    ax.set_xlabel("severity (0 = clean)"); ax.set_ylabel("mask AP"); ax.set_xticks(range(6)); ax.set_ylim(0, max(0.8, clean * 1.1))
    ax.set_title(f"Robustness on val-C — mPC {mpc:.3f}, rPC {mpc/clean:.2f}"); ax.legend(frameon=False, fontsize=6.5)
    fig.tight_layout(); savefig(fig, FIG / "fig14_valC_robustness"); plt.show()
    display(VC.pivot(index="scenario", columns="level", values="mask_AP").round(4))
else:
    print("val-C not evaluated")

## 14. ความเร็วในการประมวลผล

**ชุดข้อมูล:** ภาพ 50 ภาพแรกของ val (ภาพจริง) เวลาที่วัดไม่รวม preprocessing นอกแบบจำลอง (เช่น unsharp masking) ซึ่งต้องบวกเพิ่มเมื่อใช้งานจริง


วัดเวลาของ pipeline ของ Ultralytics (preprocess, inference และ postprocess) ด้วยภาพจริงจากชุด validation จำนวน 50 ภาพที่ batch 1 บน GPU ที่ใช้ฝึก หลังจาก warm-up 10 ภาพ ค่านี้เป็นเวลาของ GPU สำหรับการพัฒนา ผลบนอุปกรณ์ปลายทางรายงานไว้ใน Notebook 03 หัวข้อ 10

In [ ]:
SPEED = pd.DataFrame(columns=["p50_ms", "p95_ms", "mean"], index=["total"], dtype=float)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
if RUN_SPEED:
  pm = YOLO(str(WEIGHTS))
  for p in VAL.path.head(10):
    pm.predict(p, imgsz=IMGSZ, verbose=False)
  sp = pd.DataFrame([pm.predict(p, imgsz=IMGSZ, verbose=False)[0].speed for p in VAL.path.head(50)])
  SPEED = sp.describe(percentiles=[0.5, 0.95]).T[["50%", "95%", "mean"]].rename(columns={"50%": "p50_ms", "95%": "p95_ms"})
  SPEED.loc["total"] = sp.sum(axis=1).describe(percentiles=[0.5, 0.95])[["50%", "95%", "mean"]].values
  SPEED["fps_from_p50"] = 1000 / SPEED.p50_ms
  save_table(SPEED.reset_index().rename(columns={"index": "stage"}), TAB / "speed_gpu_bs1.csv")
  print("device:", gpu); display(SPEED.round(2))
else:
  print("speed not measured (RUN_SPEED = False)")

## 15. การวิเคราะห์ชุด test (เฉพาะเมื่อ Notebook 03 เปิดชุด test แล้ว)

**ชุดข้อมูล:** test ทั้งชุด โดยอ่าน detection จาก `outputs/03_yolo_experiments/evaluation/dets_test.json.gz` ที่ Notebook 03 บันทึกไว้ ไม่ทำนายใหม่


**รอบที่สอง (post-hoc):** ชุด test ถูกเปิดครั้งแรกในรอบที่หนึ่ง (ผลอยู่ใน `outputs/_archive_round1/`) ผลในส่วนนี้ของรอบที่สองจึงเป็นการวิเคราะห์หลังเปิดผล

ส่วนนี้ไม่เปิดชุด test เอง แต่จะทำงานเมื่อมีไฟล์ `TEST_OPENED.json` และ detection ของชุด test ที่ Notebook 03 บันทึกไว้แล้วเท่านั้น โดยใช้ threshold ที่ตรึงไว้และการวิเคราะห์ชุดเดียวกับข้างต้น ผลในส่วนนี้มีไว้รายงาน และต้องไม่ใช้ปรับการตั้งค่าใด ๆ

In [ ]:
TEST_FLAG = ROOT / "outputs" / "03_yolo_experiments" / "evaluation" / "TEST_OPENED.json"
TEST_DETS = ROOT / "outputs" / "03_yolo_experiments" / "evaluation" / "dets_test.json.gz"
if FROZEN is not None and TEST_FLAG.exists() and TEST_DETS.exists():
    opened = json.loads(TEST_FLAG.read_text()); assert opened["weights_sha256"] == FROZEN["weights_sha256"], "test detections from a different model"
    TEST = split_images("test"); GT_TEST = DEV_DIR / "annotations" / "test_coco.json"
    with gzip.open(TEST_DETS, "rt") as f:
        DETS_TEST = json.load(f)
    COCO_TEST, T_TEST = coco_block(GT_TEST, DETS_TEST, TEST, "test")
    display(T_TEST[T_TEST.protocol == "ignore"].pivot_table(index="metric", columns="type", values="value").round(4))
    ids_test = TEST.image_id.astype(int).tolist()
    pr_figure(COCO_TEST, COCO_TEST["segm"]["recs"], ids_test, CONF_TH, "test")
    D_T, G_T, CACHE_T = build_matches(GT_TEST, DETS_TEST, TEST); CACHE_ALL["test"] = CACHE_T
    DM_T, GM_T = class_aware_match(D_T, G_T, CACHE_T, CONF_TH)
    CM_T = confusion_instance(D_T, G_T, CACHE_T, CONF_TH); save_table(CM_T.reset_index(), TAB / "test_confusion_instance.csv")
    plot_confusion(CM_T, "Instance confusion matrix (test, frozen thresholds)", FIG / "fig04_test_confusion_instance")
    DT_T, GTT, DTK_T, GTK_T = tide_report(D_T, G_T, CACHE_T, CONF_TH, "test")
    mask_quality(DM_T, DETS_TEST, GT_TEST, "test"); size_analysis(GM_T, DM_T, "test"); calibration(D_T, G_T, CACHE_T, "test")
    per_group(COCO_TEST, DM_T, GM_T, TEST, "test"); state_eval(DETS_TEST, TEST, TH_STATE, "test")
    galleries(TEST, DM_T, GM_T, DT_T, DETS_TEST, GT_TEST, "test")
else:
    print("test set not opened by Notebook 03 (or frozen config missing) — section skipped")

## 16. สรุปผล

ตารางสรุปรวบรวมตัวชี้วัดหลักของชุด validation ไว้ในไฟล์ `outputs/03.5_best_model_evaluation/summary.json` เพื่อใช้อ้างอิงในบทผลการทดลอง รูปทั้งหมดอยู่ใน `figures/` (PNG 300 dpi และ PDF) และตารางอยู่ใน `tables/`

In [ ]:
mask_row = lambda m: float(T_VAL[(T_VAL.type == "mask") & (T_VAL.protocol == "ignore") & (T_VAL.metric == m)].value.iloc[0])
ci_row = T_VAL[(T_VAL.type == "mask") & (T_VAL.protocol == "ignore") & (T_VAL.metric == "AP")].iloc[0]
tide = pd.read_csv(TAB / "val_tide_errors.csv")
SUMMARY = dict(model=MODEL_NAME, provisional=PROVISIONAL, round=CARD["round"], model_size=CARD["model_size"], imgsz=IMGSZ, weights_sha256=CARD["weights_sha256"], params=CARD["parameters"], GFLOPs=CARD["GFLOPs@imgsz"],
               val=dict(mask_AP=mask_row("AP"), mask_AP_CI95=[float(ci_row.ci_low), float(ci_row.ci_high)], mask_AP50=mask_row("AP50"),
                        mask_AP75=mask_row("AP75"), mask_AP_gravel=mask_row("AP_gravel"), mask_AP_sand=mask_row("AP_sand"),
                        conf_thresholds=CONF_TH, tp_fp_fn=summ.set_index("cls")[["TP", "FP", "FN", "precision", "recall", "F1"]].round(4).to_dict("index"),
                        tide_delta_AP50=dict(zip(tide.error, tide.delta_AP50.round(4))), largest_error=tide.sort_values("delta_AP50").error.iloc[-1],
                        boundary_iou_median=Q_VAL.groupby("cls").boundary_iou.median().round(4).to_dict(),
                        state=json.loads((TAB / "val_state_metrics.json").read_text())),
               speed_gpu_p50_ms=float(SPEED.loc["total", "p50_ms"]), device=gpu)
(OUT_ROOT / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=float))
print(json.dumps(SUMMARY, indent=2, default=float))